# Task 10 — Final Validation, Explainability and Responsible AI
## Smart Electricity Consumption Optimizer

This separate notebook evaluates the Task 9 integrated system against the
supplied Task 10 requirements. It checks forecasting performance, complete
decision behavior, explanations, privacy/fairness/safety limits and prototype
deployment, and produces a report plus structured presentation notes.

**Run all cells.** No ontology or ZIP upload is required. Existing Task 9 models
are reused when available. **A fresh Colab runtime without saved models will,
by default, rebuild Task 9 once before evaluation; this can take several minutes
and a GPU helps.** Set `PREPARE_MODELS_IF_MISSING=False` to forbid preparation.
Evaluation starts only after preparation ends and the models are fingerprinted.
No model fitting/tuning is allowed during the evaluation phase.

The earlier held-out test data is unseen to training/model selection but has
already been evaluated in Task 9. Reusing it is reported explicitly, not presented
as an independent new test. Additional household scenarios are designed tests.
An optional later-time-range minute CSV supports genuinely new supplied readings.
No new independent measured electricity data is claimed when no CSV is supplied.

In [ ]:
import importlib.util
import subprocess
import sys
requirements = {"tensorflow": "tensorflow==2.20.0", "shap": "shap>=0.48,<0.51", "ipywidgets": "ipywidgets>=8,<9",
    "numpy": "numpy>=1.26,<3", "pandas": "pandas>=2.2,<3", "sklearn": "scikit-learn>=1.5,<2",
    "matplotlib": "matplotlib>=3.8,<4", "joblib": "joblib>=1.4,<2"}
missing = [spec for name, spec in requirements.items() if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
if "google.colab" in sys.modules:
    from google.colab import output
    output.enable_custom_widget_manager()

## 1. Configuration and data provenance
Use the same Task 7 folder and Task 9 model-cache root to evaluate the exact
previously trained system. A saved notebook's visible results do not contain
its trained model weights. If setup rebuilds the models, the report labels that
fact; it does not claim to restore the exact historical run from its screenshots.

In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
import json
import time
import tempfile
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, HTML

LIVE_TASK9_BRIDGE = globals().get("bridge")  # Reused if these cells follow Task 9 in the same runtime.
TASK7_FOLDER = ""
TASK9_OUTPUT_DIR = Path("task9_outputs")  # Existing cache root, optionally a saved Drive folder.
PREPARE_MODELS_IF_MISSING = True
BUILD_TASK7_IF_MISSING = PREPARE_MODELS_IF_MISSING
TRAIN_EXAMPLES = 20000
VALIDATION_EXAMPLES = 5000
UNSEEN_MINUTE_CSV = ""  # Optional path; see the required schema below.
SCENARIO_HISTORY_EXAMPLES = 4
SHAP_SUMMARY_EXAMPLES = 24
BOOTSTRAP_REPLICATES = 200
SHOW_DASHBOARD = True
OUTPUT_DIR_10 = Path("task10_outputs") / datetime.now(timezone.utc).strftime("evaluation_%Y%m%dT%H%M%S_%fZ")
OUTPUT_DIR_10.mkdir(parents=True, exist_ok=False)

## 2. Embedded system and evaluation code
Collapsed code cells contain the supplied ontology, exact Task 7 source, Task 9
implementation and evaluation helpers. The ontology/source hashes are checked.
The ontology adapter remains the explicitly documented supported SWRL subset.

In [ ]:
import base64, hashlib, zlib
ONTOLOGY_XML = zlib.decompress(base64.b64decode('eNrtXFtz4jYUfs+vcNmX9gHMZbfbMFl2MmHTZDYbGDbtdvqSke0DqLUljySH8O8r2QZsbHJpYssE+wESW5fvnPOdoyNZ4uTzvecad8A4puRTo9NqNz4Pjk6YM+1PhueGfEj4p8ZcCL9vmovFosXBQ0RgewFWi7KZOUMBQ3f2nGIxZ0g+wyYlgrp0hoGb3Xb3V/OjGcgawgWnGT9aNrtm48hQl+yhbyEORXdCeJ8u3FQvi17YeLfd7prtj6Z8/C5VXiohr3zn+PjYlFW63aYs0eRLItB9k/B0ZfmZV/mvb1eqgd9MgjzgPrIhXYs7OyB2VNXv9lxqJoOS76jTNmW1EGNOPb5gu9TRMzsdUz3P1kDrKuq/FheITClzWuAESZP0Wr2wAGpltKpuW491bKXr5Ji2EL68awzCbk8k6v4ovm0oZ0AWDUQhfTbMqM+jo6jrn5pNI/zLLOaK246/jFNCqEBCer/hM+oDExJuukxxMIxmc3CUkf75BMN8ErjwhSBL6nbTaGjGjYDjSL48g/7fvjQbb2T9A7YwxtUy3Kt65BzxUzukZ9qskegPmLQYFHGAWMNwwGdgIyFppyA4SFJt6cMTgrhFqQtINihYACdmurE4DJlZSQdlaX0INuYV0PsKR1Lzaszr88BawRhNQyAMOA2YDY+N8oL6aTFWbrxp3KEewmR3q68m4anvuxgRmQdkMDBEZlAGhIjZZgVI94XcYUaJB0To5l0CSk29oiAklIzcM0ocXBkqXtCAw5y6jm4iroHUNCwKQqjiC6XiKjDvBjE8neqmXYSi5lxREGL9PotwJU4thjKTrdqM8NVT+hULrpEHW/42jDP5MmcYSTiNPfSN/IkOFwyTWZro29otLbaObDvwpXBL/eZeQ3ndGKsk2esIm88iTATMgFWERmO6ADZBQhJbP5ESYGoq7R+VGKYMiwoEpBWSmkT7Nqp9l/0GXD+DIhw1f/aNP9FcpCosSqKpubR3XAJPdihtxyowpUqAqZn0BCY5YGMPuVWgEubnLtxjy9VNow2QmkLPeK9ZCQqNAf17IZFrp9AKSE2h4ihU4uromYs4f4urorl7LUJpi/aR9FvowuVcUVaLqNv+UrS0Xwiw2fKMEh54vi4DZ0CUJ33eS2Y9KnjodXfhepAhS95GrlbyZ1HkjYkhlIcGxOKcMRpuQgDl2GWVXmo1SwbEoVvlAs/mqxVRrZbJBXLw1lnt2NBike3tIkVLOwGbenLUcJCukSuNoCy5c3fClCJvendG4XJiD8YgfVwLnTe9lyXvHxzYmMEUGGgKqmkEmrfxX8pE8A47AXLf4IxyCC5a3ubOK9U+F2cjfNE2TyLZGj/7amVB01brLS2U44FfAfzRdSXMkoJy6HaZwJThmXplQdltR69h0lh0WWbX2mJZAHLmq0kkOVX6ydMzxQNM+49+Ct8EjIzOzysRW9JYDj24/EB8jsnsG7LlF9zeABd6zZMD6FDDTHb9pVpRJpU8PY/KJWbS6miuhhw6JMoQuM1wtLxerN+Y6Hlekney/085wVVHmDe23JJhkCtZot/wSHQ/ffT6ZWdgc1rcXuaKViHEYD0qG2jlMNzgcxq4EgSE7z1H161QqE2lrbZcZIE7yI7vt9GIGleOSr1U3Zeen3JodbNvUWe5ufewul/Y/6mg3hWW4dXc0eQUMy6yz54G6oXgwhm/QpiHLq0yWxWVE3cHq3PS2tO/fJCIzQJFuE7x+GQo2AFnh0Nni+yw/EmMXTxqy50/iEKwu43uAVSR/285RaTQOSCndhQZRdaj7GoXxNO8Jj7HtSzTcZJH9/fDY7Jwurqmb2/FmTeOu2f5xXrvYSK9sBExLDAclQmD86TkIpPV34Z5dJ1a1KnFI1O+OrOoM4s6s6gzi1desjmIxCK6NRmeD47kFa5B/Q5EvciQKYS1NMQcjNGPK+N0fGn8HP/2pfG+9aHVPW5Jfb5vtj80O72bTrff6fTbvb9/CVdjuZRnJm0QWC055qulL5urT+TjaCXoP0rI38s=')).decode("utf-8")
assert hashlib.sha256(ONTOLOGY_XML.encode("utf-8")).hexdigest() == '2ada3ac3ccafb385f087d822ddd6dadd9a16b76bbecffc1c58cddbfa28757320'
TASK7_SOURCE_TEXT = zlib.decompress(base64.b64decode('...')).decode("utf-8")
assert hashlib.sha256(TASK7_SOURCE_TEXT.encode("utf-8")).hexdigest() == 'e4d0a2d6d84cdbc11b6ba71d1733d84fce6747b5357b0955d2f2623d9a2fc960'
TASK7_PREPROCESSOR_SOURCE = zlib.decompress(base64.b64decode('eNq1Gdtu2zb0PV9BeMAgb7Yae262BvOAoMu2Auk6JO6AIQgI2qJtrhLpklScrOi/7/AmUbKcS+f5IbGoc7/zOKNLtJF0QyTFRZlrdkskI5omBeOlpngpSUEHaEmJLiVVA1SQO7wUcktkhpcsz6eT/ukRgg9bIi40ivFSxjN6lzKFS84+lhQJ+QBMIeCd4GyBGV9IShTjK0fafCRhiqI/SV7ScymFTHpvLRUEkBlAKlSUSqM1uaXIcRsgJaSmGdKsoEqTYqPSXv/IUowlQNPG43XQ9SZdiM194jHmjKsWZApAQDWnSW/0El70BmiRC0WzaS+nSw2POZnTfNqTbLXWwNpypsQSMvRS85C484Uoua5e2KfAWswVlbegyNRDpfRjMnrp8MhClyQ3khnC6XZNJU0ChgNhfFNqfGtMZxg4jHRpvJfkrGB62nZqYFxqgN9k6c9Ek1+Myon11tTxst89KGCHIAF+VbzU/gNa18ue440/+fefezeVQMHuNy0UJ30TI9ZoH16wQQs1HFdoKVH6fkOTZS6I9tp8hV4Lfks5o1wjkjOiwHIleBbN75FeU6SJXFH9YiG4pnf6BTWiEM0EBw9lNA3Wuw4Kf9g65g0jzM4ufz2f4V/Oz2bvL89BvBrNKdjA8nZ4ACmoFiHVRtjBC3rPhcgrtd9xChoUGxARZeD04UeQlOl7pNiKQ5gZNxv9L65mb9GaKS3kfaQtzelCS7YA7SrOkCoLY5lO60OS2Rwgd0xNR/0uX0gKgNzQPzrKoFoBccozInEg4WLSV6G1KKUNEFNU7MO3/sFlLnqBTo7T4wbpRoB/qsKoZ9AxVKHeKeKbFL4kY/SN+bph8N8Sf4HGk/6ghbIQyqHAl70oNU4mtvu5ONkzci+WW0o/APb3MT+Du5ddN26NDAVXr/ezTrzdDBQaolEfCIzGMXdHYC//vQRqCkYu8CbgJ21pf5qil62AcKw/D5xm01B/TFTMS5Zn2OdjshFbKkPvWgvJ/hG8alQbU9NqgDjZQhqcaZ/hSIot0gME1lmzpU48rT5iyuXB2dUM2TqEPnCx5YhoTwIyhS5Aeuh5bAUVUQ89rkuXHHqsstW1Rdm/XWFoKbuvIZpHNch3x90g44jKuhNiUkOMuyF+CIXdt8iOLhAZsdELvkLDL/54Ahf0luYQLiBg1UsOQ9v73Nk/VFj3FL+3DR3PqXEjdi6sga1zYmhwxEPQ8DqGFpxiWyj2k1/H8HorLLzajzBeH8z0l2ZQOoUMUQymQeoHFsi/DIYrhSSMYT6GD+UL+LvBzuANf0DN6LC1hXYG74Ju2drRttbupL3eAR7vBT6gka9gvBWmoboyM1Q5jMXAQwXLztYUxl0/s63JZkM5TB9bmufIBYEtP4tSSjOihGJzWL842bCRDcMd4ZaJUmEo0ME4oWi8OunvQzOhCxgKk5Voo41ejfuPsjOtoI148v34cJXmUuQ5XB/MEArRPrQ1DamyKOAeBKaH5mTf6nZBP5Cpt1A4xdaM5dHgYaMbeuI46rU2huFsEp3ZUDXN94focBIORycxaHX8yh9/rgd3bvukE8VM716olGlaqKQfXcK8sYwv/PfEAffrpu5vUT1TOGpHfjJMPgdXJhV0RNZfh7oaovlELOCu6aWFKWFy2iAW2CudPY17LAEgJVkmltPjvWK0RKm0hSvU85Qld8/U9ccp+LRbWVMNn60sICWPqNnFSxK+ok/n9piBdhCGj2rV5YvQW9vXjkblaonYmAIfvrtUOFXch6pnD6OLvH3uMOrhStY7KP0S1fL6y5k6VPVvXuXPL85fzy7fvD67CJfHq+vR6U0dh2ArZhcLsTk77u07KeNnsOb93FFrG9DvMTTlytxGOwf4X3MxB+dVUGGcr4D8Oe6Y7VqBUcHum777eyjvjnVPpjxphMnuaLSr3lMI1zE5fKZW0eT0f7LuVLvZOp4nQ8jPSf+BftLQtFGZXJ98PrvRSX9/ST9c7p+tVpKuIHFgRpkPC6ohXcL0YsQ5DB+gbUljLbRdLu6pmS7xrso5DpLgUZTo3z4KPX4W9Hceuhksvow0ZW75q/ny0Yhogj+hYDxE/8H8spT/o+guxb5AhI40e44oh8m0L+H4tGSbuRVOWBhWiwREZFjXQH8j2S3hC7+zDdsOt9T4W8CEVMmyu3iMVx8pyey8uWR3SbjK9BoC+W2jRXJLK7jesQx7YCDDFkD0qJVlA//zgN1sxUeKfiwpSI5BrJVeT8NgX+27jvzk7hiYDs03KbHjW60VIMfrsmi9l5nN2xRQoNidTGJNwAyGWqA7RFGYOdHglWVoIYdtUb09xBYXTCl3oeiuML+9uZq9u/wLv/n9j/cz/Prdxfu3v19FwxhTnMRzF+H3YZdcH2qBeVnYn3JqQLdXDNo1HLUoixKKill+WJNJfF0hHtf2MRvPsoCrYhJp4gj5yWMuStAfiw9mrnEmgfuKXz4rsqTYnjouEM+sAGIOcICO+zUYmLEJZOxqQaJLpOMTxWulxnUgchPVi/Zry9fXVzStxAyBZ4kbc6kl43AxjPjEodk2NuRFnlcuqZcbYb/KtKL50v12N6doRXnJOM3vzdpJQZJlaRTCTog4PuoLbvXjh3NALIeL2RzK9jynDRNVDqpOvq6NGZ3VRogOK5k6ktynxnXgevMv9Qiwsw==')).decode("utf-8")
assert hashlib.sha256(TASK7_PREPROCESSOR_SOURCE.encode("utf-8")).hexdigest() == '3cb24de1b471e3f22d194ccd8f22c8646dea19dcd9c04d0a9629a37e797b88ba'

In [ ]:
"""Task 9: ontology → LSTM features → ML forecast → rules → A* scheduling.

The ontology adapter executes this project's named-subclass and unary class-to-
decision SWRL subset. It is not a general OWL DL consistency reasoner.
"""
import hashlib
import heapq
import json
import math
from dataclasses import asdict, dataclass, replace
from itertools import count
from pathlib import Path
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd


def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def local_name(iri):
    return str(iri).rsplit("}", 1)[-1].rsplit("#", 1)[-1].rsplit("/", 1)[-1]


class OntologyKnowledge:
    NS = {"rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
          "rdfs": "http://www.w3.org/2000/01/rdf-schema#", "owl": "http://www.w3.org/2002/07/owl#",
          "swrl": "http://www.w3.org/2003/11/swrl#", "swrla": "http://swrl.stanford.edu/ontologies/3.3/swrla.owl#"}

    def __init__(self, xml_text):
        self.digest = hashlib.sha256(xml_text.encode("utf-8")).hexdigest()
        root = ET.fromstring(xml_text)
        resource = "{" + self.NS["rdf"] + "}resource"
        about = "{" + self.NS["rdf"] + "}about"
        self.parents, self.individual_types, self.asserted_decisions = {}, {}, {}
        self.classes = set()
        for element in root.findall("owl:Class", self.NS):
            name = local_name(element.attrib[about])
            self.classes.add(name)
            self.parents[name] = {local_name(n.attrib[resource]) for n in element.findall("rdfs:subClassOf", self.NS)
                                  if resource in n.attrib}
        for element in root.findall("owl:NamedIndividual", self.NS):
            name = local_name(element.attrib[about])
            self.individual_types[name] = {local_name(n.attrib[resource]) for n in element.findall("rdf:type", self.NS)}
            self.asserted_decisions[name] = {local_name(n.attrib[resource]) for n in element
                                             if local_name(n.tag) == "hasDecision" and resource in n.attrib}
        self.rules = []
        for element in root:
            types = {n.attrib.get(resource) for n in element.findall("rdf:type", self.NS)}
            if self.NS["swrl"] + "Imp" not in types and element.tag != "{" + self.NS["swrl"] + "}Imp":
                continue
            enabled = element.find("swrla:isRuleEnabled", self.NS)
            if enabled is not None and (enabled.text or "").strip().lower() == "false":
                continue
            label_node = element.find("rdfs:label", self.NS)
            label = label_node.text if label_node is not None else f"SWRL_{len(self.rules)+1}"
            body = self._atoms(element.find("swrl:body", self.NS))
            head = self._atoms(element.find("swrl:head", self.NS))
            if not body or not head or any(a["kind"] != "ClassAtom" for a in body):
                raise ValueError(f"Unsupported SWRL body in {label}; this adapter will not silently ignore it.")
            variables = {a["arg1"] for a in body}
            if len(variables) != 1 or any(a["kind"] != "IndividualPropertyAtom" or a["predicate"] != "hasDecision"
                                           or a["arg1"] not in variables for a in head):
                raise ValueError(f"Unsupported SWRL head in {label}.")
            self.rules.append({"id": label, "classes": [a["predicate"] for a in body],
                               "decisions": [a["arg2"] for a in head]})
        if not self.rules:
            raise ValueError("The ontology contains no supported enabled SWRL rules.")

    def _atoms(self, container):
        if container is None or not len(container):
            return []
        resource = "{" + self.NS["rdf"] + "}resource"
        current, result = container[0], []
        while current is not None:
            first = current.find("rdf:first", self.NS)
            if first is None or not len(first):
                raise ValueError("Unsupported SWRL RDF list encoding.")
            atom = first[0]
            kind = atom.find("rdf:type", self.NS)
            kind = local_name(kind.attrib[resource]) if kind is not None else local_name(atom.tag)
            def value(name):
                node = atom.find("swrl:" + name, self.NS)
                return local_name(node.attrib[resource]) if node is not None and resource in node.attrib else None
            result.append({"kind": kind, "predicate": value("classPredicate") or value("propertyPredicate"),
                           "arg1": value("argument1"), "arg2": value("argument2")})
            rest = current.find("rdf:rest", self.NS)
            if rest is None:
                raise ValueError("SWRL list has no terminator.")
            if rest.attrib.get(resource) == self.NS["rdf"] + "nil":
                break
            if not len(rest):
                raise ValueError("Only nested SWRL RDF lists are supported by this adapter.")
            current = rest[0]
        return result

    def infer(self, appliance):
        types = set(self.individual_types.get(appliance.ontology_id, set())) | {"Appliance"}
        if appliance.essential:
            types.add("EssentialAppliance")
        if appliance.flexible:
            types.add("FlexibleAppliance")
        if appliance.priority == "High":
            types.add("HighPriorityAppliance")
        changed = True
        while changed:
            expanded = types | set().union(*(self.parents.get(t, set()) for t in types))
            changed = expanded != types
            types = expanded
        decisions, fired = set(), []
        for rule in self.rules:
            if set(rule["classes"]) <= types:
                decisions.update(rule["decisions"])
                fired.append("OWL:" + rule["id"])
        return {"types": sorted(types), "candidate_decisions": sorted(decisions), "fired": fired,
                "asserted_decisions": sorted(self.asserted_decisions.get(appliance.ontology_id, set()))}


@dataclass(frozen=True)
class Appliance:
    name: str
    power_w: float
    duration_min: int = 60
    requested: bool = True
    on: bool = False
    essential: bool = False
    flexible: bool = True
    priority: str = "Medium"
    earliest_min: int = 0
    deadline_min: int = 360
    requires_occupancy: bool = False
    ontology_id: str = ""


@dataclass(frozen=True)
class Scenario:
    occupancy: int = 3
    temperature_c: float = 35.0
    peak_start_hour: int = 18
    peak_end_hour: int = 20
    peak_rate: float = 9.0
    offpeak_rate: float = 6.0
    horizon_hours: int = 6
    max_parallel_jobs: int = 1
    node_limit: int = 50000


def default_appliances():
    return [Appliance("Refrigerator", 200, on=True, essential=True, flexible=False,
                      priority="High", ontology_id="Refrigerator_1"),
            Appliance("Air Conditioner", 1800, on=True, flexible=False, priority="High", requires_occupancy=True),
            Appliance("Washing Machine", 500, ontology_id="WashingMachine_Test"),
            Appliance("Dishwasher", 1200)]


def validate_inputs(appliances, scenario):
    if not 0 <= scenario.occupancy <= 20 or not np.isfinite(scenario.temperature_c):
        raise ValueError("Occupancy must be 0–20 and temperature must be finite.")
    if scenario.horizon_hours not in range(1, 13) or scenario.max_parallel_jobs not in range(1, 7):
        raise ValueError("Use a 1–12 hour horizon and 1–6 concurrent schedulable jobs.")
    if not 0 <= scenario.peak_start_hour < 24 or not 0 <= scenario.peak_end_hour < 24:
        raise ValueError("Tariff hours must be 0–23.")
    if any(not np.isfinite(v) or v < 0 for v in [scenario.peak_rate, scenario.offpeak_rate]):
        raise ValueError("Tariffs must be finite and non-negative.")
    if scenario.node_limit < 1 or not 1 <= len(appliances) <= 8:
        raise ValueError("Use 1–8 appliances and a positive search limit.")
    if len({a.name for a in appliances}) != len(appliances):
        raise ValueError("Appliance names must be unique.")
    for a in appliances:
        if any(type(getattr(a, field)) is not bool for field in ["requested", "on", "essential", "flexible", "requires_occupancy"]):
            raise ValueError("Appliance flags must be JSON true/false values.")
        if any(type(getattr(a, field)) is not int for field in ["duration_min", "earliest_min", "deadline_min"]):
            raise ValueError("Runtime, earliest start and deadline must be integer minutes.")
        if not a.name.strip() or not np.isfinite(a.power_w) or not 0 < a.power_w <= 20000:
            raise ValueError("Each appliance needs a name and a power rating in (0, 20000] W.")
        if a.priority not in ["Low", "Medium", "High"]:
            raise ValueError(f"Invalid priority for {a.name}.")
        if a.duration_min <= 0 or a.duration_min % 15 or a.earliest_min < 0 or a.earliest_min % 15 or a.deadline_min % 15:
            raise ValueError("Runtime, earliest start and deadline must use 15-minute steps.")
        if a.deadline_min <= a.earliest_min:
            raise ValueError(f"Deadline must follow earliest start for {a.name}.")


def peak_at(timestamp, scenario):
    hour = timestamp.hour + timestamp.minute / 60
    start, end = scenario.peak_start_hour, scenario.peak_end_hour
    if start == end:
        return False
    return start <= hour < end if start < end else hour >= start or hour < end


def rate_at(timestamp, scenario):
    return scenario.peak_rate if peak_at(timestamp, scenario) else scenario.offpeak_rate


def reason_appliance(appliance, scenario, origin, forecast_kw, threshold, knowledge):
    inferred = knowledge.infer(appliance)
    essential = "EssentialAppliance" in inferred["types"]
    flexible = appliance.flexible and "FlexibleAppliance" in inferred["types"]
    high_priority = "HighPriorityAppliance" in inferred["types"]
    trace = list(inferred["fired"])
    # Imported SWRL heads propose decisions. Ordered operational rules resolve
    # conflicts and add the tariff, occupancy, comfort and forecast conditions.
    if essential:
        action, reason, kind, rule = "Keep ON", "Essential appliance; protected regardless of tariff or forecast.", "protected", "R1-essential"
    elif scenario.occupancy == 0 and appliance.requires_occupancy:
        action, reason, kind, rule = "Turn OFF" if appliance.on else "Do not start", "Unoccupied home; this appliance requires occupancy.", "inactive", "R5-empty-home"
    elif appliance.name == "Air Conditioner" and scenario.occupancy > 0 and scenario.temperature_c >= 38:
        action, reason, kind, rule = "Keep ON", "Occupied home at or above 38°C; comfort overrides cost saving.", "protected", "R2-comfort"
    elif not appliance.requested and not appliance.on:
        action, reason, kind, rule = "Leave OFF", "Appliance was not requested.", "inactive", "R6-not-requested"
    elif high_priority or appliance.on:
        action, reason, kind, rule = "Run as scheduled", "High-priority or already running appliance; no automatic interruption or delay.", "protected", "R3-priority"
    elif flexible:
        high = forecast_kw >= threshold
        action, reason, kind, rule = "Find schedule", (
            "Flexible job; " + ("avoid the single forecast high-demand interval; " if high else "") +
            ("current tariff is peak; " if peak_at(origin, scenario) else "") + "search permitted start times for lowest estimated job cost."), "job", "R4-forecast-tariff"
    else:
        action, reason, kind, rule = "Run at earliest allowed time", "Non-flexible requested job keeps its earliest start.", "job", "R6-fixed-job"
    trace.append(rule)
    return {"Appliance": appliance.name, "Action": action, "Reason": reason, "Kind": kind,
            "Essential": essential, "Flexible": flexible, "Rule_trace": trace,
            "OWL_candidates": inferred["candidate_decisions"], "OWL_asserted_decisions": inferred["asserted_decisions"],
            "Ontology_types": inferred["types"]}


def plan_jobs(jobs, scenario, origin, target_end, forecast_high):
    """A* assignment search; admissible remaining-cost lower bound.

    Capacity counts only these schedulable jobs, not whole-house load. Forecast
    avoidance applies only to the known 15-minute target interval.
    """
    slots = scenario.horizon_hours * 4
    rates = np.array([rate_at(origin + pd.Timedelta(minutes=15 * i), scenario) for i in range(slots)])
    options = []
    for job, flexible in jobs:
        duration = job.duration_min // 15
        latest = min(slots, job.deadline_min // 15) - duration
        starts = list(range(job.earliest_min // 15, latest + 1))
        if not flexible:
            starts = starts[:1]
        choices = []
        for start in starts:
            begin, end = origin + pd.Timedelta(minutes=15 * start), origin + pd.Timedelta(minutes=15 * (start + duration))
            overlaps_forecast = begin < target_end and end > target_end - pd.Timedelta(minutes=15)
            if flexible and forecast_high and overlaps_forecast:
                continue
            cost = float((job.power_w / 1000) * .25 * rates[start:start + duration].sum())
            choices.append((start, duration, cost))
        if not choices:
            return {"status": "infeasible", "reason": f"No allowed start for {job.name}; check deadline and forecast avoidance.",
                    "schedule": [], "estimated_cost": None, "baseline_cost": None, "estimated_saving": None, "nodes": 0}
        options.append(choices)
    if not jobs:
        return {"status": "optimal", "reason": "No schedulable jobs requested.", "schedule": [],
                "estimated_cost": 0., "baseline_cost": 0., "estimated_saving": 0., "nodes": 0}
    # A chronological DFS supplies a feasible earliest-start comparison under
    # exactly the same constraints. It is not claimed to be the actual old bill.
    baseline_nodes = 0
    baseline_limit = False
    def first_feasible(index, loads, starts):
        nonlocal baseline_nodes, baseline_limit
        if baseline_nodes >= scenario.node_limit:
            baseline_limit = True
            return None
        baseline_nodes += 1
        if index == len(jobs):
            return starts
        for start, duration, cost in options[index]:
            if max(loads[start:start + duration], default=0) >= scenario.max_parallel_jobs:
                continue
            updated = list(loads)
            for i in range(start, start + duration):
                updated[i] += 1
            result = first_feasible(index + 1, updated, starts + [(start, duration, cost)])
            if result is not None:
                return result
        return None
    baseline = first_feasible(0, [0] * slots, [])
    if baseline is None:
        return {"status": "search_limit" if baseline_limit else "infeasible",
                "reason": "No feasible schedule found before the search limit." if baseline_limit else "Requested jobs cannot all meet their windows and concurrency limit.",
                "schedule": [], "estimated_cost": None, "baseline_cost": None, "estimated_saving": None, "nodes": 0}
    minimum = [min(x[2] for x in choices) for choices in options]
    lower = [sum(minimum[i:]) for i in range(len(jobs) + 1)]
    queue, counter = [], count()
    heapq.heappush(queue, (lower[0], 0., next(counter), 0, (0,) * slots, []))
    best = {(0, (0,) * slots): 0.}
    chosen, status, explored = baseline, "feasible_search_limit", 0
    while queue and explored < scenario.node_limit:
        _, cost_so_far, _, index, loads, path = heapq.heappop(queue)
        if cost_so_far > best.get((index, loads), float("inf")) + 1e-10:
            continue
        explored += 1
        if index == len(jobs):
            chosen, status = path, "optimal"
            break
        for start, duration, cost in options[index]:
            if max(loads[start:start + duration], default=0) >= scenario.max_parallel_jobs:
                continue
            updated = list(loads)
            for i in range(start, start + duration):
                updated[i] += 1
            new_cost, state = cost_so_far + cost, (index + 1, tuple(updated))
            if new_cost + 1e-10 < best.get(state, float("inf")):
                best[state] = new_cost
                heapq.heappush(queue, (new_cost + lower[index + 1], new_cost, next(counter),
                                       index + 1, state[1], path + [(start, duration, cost)]))
    rows = []
    for (job, _), (start, duration, cost) in zip(jobs, chosen):
        rows.append({"Appliance": job.name, "Start": str(origin + pd.Timedelta(minutes=15 * start)),
                     "End": str(origin + pd.Timedelta(minutes=15 * (start + duration))),
                     "Power_W": job.power_w, "Energy_kWh": job.power_w / 1000 * duration * .25,
                     "Estimated_cost": cost})
    total, baseline_cost = sum(p[2] for p in chosen), sum(p[2] for p in baseline)
    return {"status": status, "reason": "Minimum cost under declared constraints." if status == "optimal" else "Feasible DFS schedule retained; A* reached its node limit, so optimality is unproven.",
            "schedule": rows, "estimated_cost": total, "baseline_cost": baseline_cost,
            "estimated_saving": baseline_cost - total, "nodes": explored}


def optimize_decisions(forecast, appliances, scenario, knowledge):
    validate_inputs(appliances, scenario)
    origin, target = pd.Timestamp(forecast["origin"]), pd.Timestamp(forecast["target_end"])
    value, threshold = float(forecast["forecast_kw"]), float(forecast["threshold_kw"])
    if not np.isfinite([value, threshold]).all() or value < 0 or threshold <= 0:
        raise ValueError("Forecast and training threshold must be finite valid power values.")
    if target - origin != pd.Timedelta(hours=1):
        raise ValueError("Expected a one-hour-ahead forecast target.")
    rows = [reason_appliance(a, scenario, origin, value, threshold, knowledge) for a in appliances]
    jobs = [(a, row["Flexible"]) for a, row in zip(appliances, rows) if row["Kind"] == "job"]
    plan = plan_jobs(jobs, scenario, origin, target, value >= threshold)
    for row in rows:
        if row["Kind"] != "job":
            continue
        scheduled = next((s for s in plan["schedule"] if s["Appliance"] == row["Appliance"]), None)
        if scheduled is None:
            row["Action"] = "No feasible schedule — review inputs"
            row["Suggested_start"] = None
        else:
            row["Suggested_start"] = scheduled["Start"]
            row["Action"] = "Delay" if pd.Timestamp(scheduled["Start"]) > origin else "Run now"
    return {"forecast": forecast, "scenario": asdict(scenario), "appliances": [asdict(a) for a in appliances],
            "recommendations": rows, "plan": plan, "ontology_sha256": knowledge.digest,
            "provenance": "Electricity history is recorded data; household settings are user-supplied scenarios. Savings concern requested schedulable jobs only, not measured whole-house bills."}


class ForecastBridge:
    """Frozen Task 7 LSTM features + validation-selected Task 6 algorithm family."""
    def __init__(self, artifact_dir, output_root, train_cap=20000, validation_cap=5000, force_fit=False):
        import joblib
        from tensorflow import keras
        from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
        self.folder, self.output_root = Path(artifact_dir), Path(output_root)
        self.manifest = json.loads((self.folder / "manifest.json").read_text(encoding="utf-8"))
        if self.manifest["run_mode"] != "full" or self.manifest["horizon_steps"] != 4 or self.manifest["sequence_length"] != 96:
            raise ValueError("Use a full updated Task 7 run with 96 steps and a four-step horizon.")
        self.network = keras.models.load_model(self.folder / "task7_FINAL_lstm.keras", compile=False)
        self.scaler = joblib.load(self.folder / "power_scaler.joblib")
        self.mean, self.scale = float(self.scaler.mean_[0]), float(self.scaler.scale_[0])
        with np.load(self.folder / "prepared_arrays.npz", allow_pickle=False) as archive:
            self.arrays = {key: archive[key] for key in archive.files}
        self.history = self.arrays["history_values"]
        self.context = self.arrays["context_values"]
        self.timestamps = pd.DatetimeIndex(pd.to_datetime(self.arrays["timestamps_ns"]))
        self.y = self.arrays["target_scaled"].reshape(-1) * self.scale + self.mean
        self.train_ids = self._sample(self.arrays["train_indices"], train_cap)
        self.validation_ids = self._sample(self.arrays["validation_indices"], validation_cap)
        self.test_ids = self.arrays["test_indices"]
        if not len(self.train_ids) or not len(self.validation_ids) or not len(self.test_ids):
            raise ValueError("All temporal partitions must be non-empty.")
        assert self.train_ids.max() < self.validation_ids.min() - 4
        assert self.validation_ids.max() < self.test_ids.min() - 4
        self.encoder = keras.Model(self.network.inputs, self.network.get_layer("lstm_summary").output)
        self.input_order = [tensor.name.split(":")[0] for tensor in self.network.inputs]
        self.feature_names = [f"LSTM_pattern_{i+1:02d}" for i in range(int(self.encoder.output_shape[-1]))] + ["Context: " + n for n in self.manifest["context_features"]]
        self.signature = hashlib.sha256(json.dumps({"model": file_hash(self.folder / "task7_FINAL_lstm.keras"),
            "arrays": file_hash(self.folder / "prepared_arrays.npz"), "manifest": file_hash(self.folder / "manifest.json"),
            "scaler": file_hash(self.folder / "power_scaler.joblib"),
            "train_cap": train_cap, "validation_cap": validation_cap, "schema": 1}, sort_keys=True).encode()).hexdigest()
        self.cache = self.output_root / "model_cache" / self.signature[:16]
        self.cache.mkdir(parents=True, exist_ok=True)
        cache_file = self.cache / "hybrid_forecaster.joblib"
        if cache_file.is_file() and not force_fit:
            saved = joblib.load(cache_file)
            if saved["signature"] != self.signature:
                raise ValueError("Hybrid-model cache does not match this Task 7 run.")
            self.regressor, self.selected_name = saved["model"], saved["selected_name"]
            self.validation_results = pd.DataFrame(saved["validation_results"])
            print("Reusing cached ML prediction stage:", self.selected_name)
        else:
            print(f"Extracting frozen LSTM patterns: {len(self.train_ids)} training / {len(self.validation_ids)} validation examples...")
            train_z, val_z = self.features(self.train_ids), self.features(self.validation_ids)
            candidates = {"LSTM + Gradient Boosting": GradientBoostingRegressor(n_estimators=100, max_depth=3, learning_rate=.05, random_state=42),
                          "LSTM + Random Forest": RandomForestRegressor(n_estimators=60, max_depth=12, min_samples_leaf=4, random_state=42, n_jobs=2)}
            records = []
            for name, model in candidates.items():
                print("Fitting ML stage:", name, flush=True)
                model.fit(train_z, self.y[self.train_ids])
                prediction = np.maximum(model.predict(val_z), 0)
                records.append({"Model": name, **self.metrics(self.y[self.validation_ids], prediction)})
            self.validation_results = pd.DataFrame(records).sort_values("RMSE_kW").reset_index(drop=True)
            self.selected_name = self.validation_results.iloc[0]["Model"]
            self.regressor = candidates[self.selected_name]
            joblib.dump({"signature": self.signature, "model": self.regressor, "selected_name": self.selected_name,
                         "validation_results": self.validation_results.to_dict("records")}, cache_file)
        self.validation_results.to_csv(self.cache / "validation_comparison.csv", index=False)
        self.threshold = float(self.manifest["peak_threshold_kw"])

    @staticmethod
    def _sample(ids, cap):
        ids = np.asarray(ids, dtype=np.int64)
        return ids[np.unique(np.linspace(0, len(ids) - 1, min(len(ids), cap), dtype=int))]

    @staticmethod
    def metrics(actual, predicted):
        from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
        return {"MAE_kW": float(mean_absolute_error(actual, predicted)),
                "RMSE_kW": float(np.sqrt(mean_squared_error(actual, predicted))), "R2": float(r2_score(actual, predicted))}

    def inputs(self, ids):
        ids = np.asarray(ids, dtype=np.int64)
        positions = ids[:, None] - 4 - 96 + 1 + np.arange(96)[None, :]
        if positions.min() < 0:
            raise ValueError("Not enough history for this forecast.")
        result = {"history": np.asarray(self.history[positions], dtype=np.float32),
                  "context": np.asarray(self.context[ids], dtype=np.float32)}
        if not all(np.isfinite(v).all() for v in result.values()):
            raise ValueError("A requested forecast contains unavailable input features.")
        return result

    def features(self, ids, batch_size=256):
        batches = []
        for offset in range(0, len(ids), batch_size):
            x = self.inputs(ids[offset:offset+batch_size])
            latent = np.asarray(self.encoder([x[key] for key in self.input_order], training=False))
            batches.append(np.concatenate([latent, x["context"]], axis=1))
        return np.concatenate(batches)

    def predict(self, target_index):
        target_index = int(target_index)
        if target_index not in self.test_ids:
            raise ValueError("Dashboard examples must come from the held-out test period.")
        z = self.features(np.array([target_index]))
        raw = float(self.regressor.predict(z)[0])
        return {"origin": str(self.timestamps[target_index - 4]), "target_end": str(self.timestamps[target_index]),
                "target_interval_start": str(self.timestamps[target_index] - pd.Timedelta(minutes=15)),
                "forecast_kw": max(raw, 0.), "raw_forecast_kw": raw, "threshold_kw": self.threshold,
                "model": self.selected_name, "target_index": target_index,
                "units": "kW, average over the target 15-minute interval"}

    def evaluate(self):
        from tensorflow import keras
        ids = self.test_ids
        hybrid = np.maximum(self.regressor.predict(self.features(ids)), 0)
        saved = pd.read_csv(self.folder / "test_predictions.csv", parse_dates=["Target_interval_end"])
        matched = saved.set_index("Target_interval_end").loc[self.timestamps[ids]]
        np.testing.assert_allclose(self.y[ids], matched["Actual_kW"], atol=1e-5)
        predictions = {self.selected_name: hybrid,
            "Task 7 LSTM": matched[self.manifest["selection"]["selected_lstm"] + "_kW"].to_numpy(),
            "Basic neural network": matched["basic_mlp_kW"].to_numpy(), "Persistence": matched["Persistence_kW"].to_numpy()}
        records = []
        for segment, mask in {"All test": np.ones(len(ids), dtype=bool), "High demand": self.y[ids] >= self.threshold}.items():
            for name, values in predictions.items():
                metrics = self.metrics(self.y[ids][mask], values[mask]) if mask.sum() >= 2 else {"MAE_kW": None, "RMSE_kW": None, "R2": None}
                records.append({"Model": name, "Segment": segment, "Samples": int(mask.sum()), **metrics})
        result = pd.DataFrame(records)
        result.to_csv(self.cache / "full_test_comparison.csv", index=False)
        return result

    def explain(self, target_index):
        import shap
        z = self.features(np.array([target_index]))
        explainer = shap.TreeExplainer(self.regressor)
        values = np.asarray(explainer.shap_values(z)).reshape(-1)
        base = float(np.asarray(explainer.expected_value).reshape(-1)[0])
        raw = float(self.regressor.predict(z)[0])
        np.testing.assert_allclose(base + values.sum(), raw, rtol=1e-5, atol=1e-5)
        return {"values": values, "base": base, "raw_prediction": raw, "data": z[0], "features": self.feature_names}

    def activation(self, target_index):
        from tensorflow import keras
        x = self.inputs(np.array([target_index]))
        view = keras.Model(self.network.inputs, self.network.get_layer("lstm_temporal_features").output)
        return np.asarray(view([x[k] for k in self.input_order], training=False))[0]


def find_task7_folder(explicit="", roots=None):
    required = ["manifest.json", "task7_FINAL_lstm.keras", "task7_basic_mlp.keras", "power_scaler.joblib",
                "prepared_arrays.npz", "test_predictions.csv"]
    def complete(folder):
        try:
            return all((folder / name).is_file() for name in required) and json.loads((folder / "manifest.json").read_text(encoding="utf-8"))["run_mode"] == "full"
        except (ValueError, KeyError):
            return False
    if explicit:
        folder = Path(explicit).expanduser()
        if not complete(folder):
            raise ValueError(f"Incomplete full Task 7 output folder: {folder}")
        return folder.resolve()
    folders = set()
    for root in roots or [Path.cwd()]:
        root = Path(root)
        if complete(root):
            folders.add(root.resolve())
        for manifest in (root / "task7_FINAL_outputs").glob("*/manifest.json"):
            if complete(manifest.parent):
                folders.add(manifest.parent.resolve())
    return max(folders, key=lambda folder: (folder / "manifest.json").stat().st_mtime) if folders else None

In [ ]:
def prepare_task7_results():
    roots = [Path.cwd()]
    for key in ["TASK7_COMPLETED_OUTPUT_DIR", "ARTIFACT_DIR", "OUTPUT_DIR"]:
        value = globals().get(key)
        if value:
            roots.append(Path(value))
    folder = find_task7_folder(TASK7_FOLDER, roots)
    if folder is not None:
        print("Reusing saved Task 7 models:", folder)
        return folder
    if not BUILD_TASK7_IF_MISSING:
        raise FileNotFoundError("No completed updated Task 7 folder is available in this runtime. Set TASK7_FOLDER, or enable BUILD_TASK7_IF_MISSING.")
    print("Fresh runtime: building Task 7 models once from the real UCI dataset. This includes model training.", flush=True)
    print("SHAP and the dashboard run after model preparation. No ZIP transfer is required.", flush=True)
    # Keras temporary model files stay within this writable project directory.
    old_tempdir = tempfile.tempdir
    tempdir = (TASK9_OUTPUT_DIR / "runtime_temp").resolve()
    tempdir.mkdir(parents=True, exist_ok=True)
    namespace = {"__name__": "task7_bootstrap"}
    try:
        tempfile.tempdir = str(tempdir)
        exec(compile(TASK7_SOURCE_TEXT, "embedded_task7_unchanged.py", "exec"), namespace)
        folder = Path(namespace["OUTPUT_DIR"]).resolve()
    finally:
        tempfile.tempdir = old_tempdir
    return find_task7_folder(str(folder))

In [ ]:
"""Task 10 evaluation helpers. Evaluation functions never fit or tune models."""
from dataclasses import asdict, replace
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import time

import numpy as np
import pandas as pd

if "Appliance" not in globals():
    from task9_core import Appliance, Scenario, default_appliances, optimize_decisions, rate_at, file_hash


def freeze_fingerprint(bridge, knowledge):
    import joblib
    weights = hashlib.sha256()
    for value in bridge.network.get_weights():
        value = np.asarray(value)
        weights.update(str((value.shape, value.dtype)).encode())
        weights.update(value.tobytes())
    return {"lstm_weights_sha256": weights.hexdigest(), "ml_model_hash": joblib.hash(bridge.regressor),
            "task7_model_file_sha256": file_hash(bridge.folder / "task7_FINAL_lstm.keras"),
            "hybrid_cache_file_sha256": file_hash(bridge.cache / "hybrid_forecaster.joblib"),
            "ontology_sha256": knowledge.digest, "hybrid_signature": bridge.signature}


def collect_frozen_predictions(bridge):
    """Same held-out inputs as Task 9; unseen to fitting, already evaluated."""
    ids = np.asarray(bridge.test_ids, dtype=int)
    train = np.asarray(bridge.arrays["train_indices"])
    validation = np.asarray(bridge.arrays["validation_indices"])
    if np.intersect1d(ids, train).size or np.intersect1d(ids, validation).size:
        raise ValueError("Test indices overlap training or validation.")
    if not (train.max() < validation.min() - 4 and validation.max() < ids.min() - 4):
        raise ValueError("Temporal partition/gap check failed.")
    z = bridge.features(ids)
    prediction = np.maximum(bridge.regressor.predict(z), 0.)
    saved = pd.read_csv(bridge.folder / "test_predictions.csv", parse_dates=["Target_interval_end"])
    aligned = saved.set_index("Target_interval_end").loc[bridge.timestamps[ids]]
    np.testing.assert_allclose(bridge.y[ids], aligned["Actual_kW"], atol=1e-5)
    frame = pd.DataFrame({"Target_index": ids, "Forecast_origin": bridge.timestamps[ids-4],
        "Target_interval_end": bridge.timestamps[ids], "Actual_kW": bridge.y[ids],
        "Hybrid_kW": prediction, "Task7_LSTM_kW": aligned[bridge.manifest["selection"]["selected_lstm"] + "_kW"].to_numpy(),
        "Basic_NN_kW": aligned["basic_mlp_kW"].to_numpy(), "Persistence_kW": aligned["Persistence_kW"].to_numpy()})
    if not np.isfinite(frame[["Actual_kW", "Hybrid_kW", "Task7_LSTM_kW", "Basic_NN_kW", "Persistence_kW"]]).all().all():
        raise ValueError("Non-finite evaluation values.")
    return frame


def external_csv_predictions(bridge, csv_path, task7_preprocessors):
    """Apply frozen Task 7 transformations to later, supplied minute readings.

    Never fit scalers. Complete observed targets are required for scored rows.
    Origins must be later than the original raw data span, not merely nontraining.
    """
    import joblib
    frame = pd.read_csv(csv_path)
    features = list(bridge.manifest["electrical_features"])
    if set(["Datetime", *features]) - set(frame.columns):
        raise ValueError("External CSV needs Datetime and all seven original electrical columns.")
    stamps = pd.to_datetime(frame["Datetime"], errors="coerce")
    if stamps.isna().any() or stamps.dt.tz is not None or stamps.duplicated().any():
        raise ValueError("Use unique valid local-time Datetime values without a timezone suffix.")
    if (stamps.dt.second.ne(0) | stamps.dt.microsecond.ne(0)).any():
        raise ValueError("External observations must lie on minute boundaries.")
    minute = frame[features].apply(pd.to_numeric, errors="coerce")
    minute = minute.where(np.isfinite(minute) & minute.ge(0))
    minute.index = pd.DatetimeIndex(stamps)
    minute = minute.sort_index()
    functions = task7_preprocessors
    power = functions["prepare_multivariate"](minute, features, 4)
    context = functions["build_context"](power, 4)
    if list(context.columns) != list(bridge.manifest["context_features"]):
        raise ValueError("External feature names/order do not match the frozen model.")
    ids = functions["valid_target_indices"](power, context, 96, 4)
    old_end = pd.Timestamp(bridge.manifest["data_audit"]["raw_end"])
    ids = ids[power.index[ids - 4] > old_end]
    if not len(ids):
        raise ValueError("No scoreable new-time-range examples. Supply at least eight days of minute readings, including warm-up, with forecast origins after the original dataset ends.")
    scalers = joblib.load(bridge.folder / "electrical_scalers.joblib")
    context_scaler = joblib.load(bridge.folder / "context_scaler.joblib")
    scaled = [scalers[name].transform(power[["input_"+name]].to_numpy()).ravel() for name in features]
    calendar = functions["calendar_features"](power.index)
    history = np.column_stack([*scaled, power["electrical_observed_fraction"].to_numpy(),
        calendar[["hour_sin", "hour_cos", "dow_sin", "dow_cos"]].to_numpy()]).astype("float32")
    expected = [*(name + "_scaled" for name in features), "electrical_observed_fraction", "hour_sin", "hour_cos", "dow_sin", "dow_cos"]
    if expected != bridge.manifest["history_features"]:
        raise ValueError("External history channels do not match the frozen model.")
    contexts = context_scaler.transform(context).astype("float32")
    rows = []
    for offset in range(0, len(ids), 256):
        part = ids[offset:offset+256]
        positions = part[:, None] - 4 - 96 + 1 + np.arange(96)[None, :]
        inputs = {"history": history[positions], "context": contexts[part]}
        if not all(np.isfinite(array).all() for array in inputs.values()):
            raise ValueError("Unavailable external inputs survived eligibility checks.")
        latent = np.asarray(bridge.encoder([inputs[k] for k in bridge.input_order], training=False))
        z = np.concatenate([latent, inputs["context"]], axis=1)
        hybrid = np.maximum(bridge.regressor.predict(z), 0)
        raw_lstm = bridge.network(inputs, training=False)
        if isinstance(raw_lstm, dict):
            raw_lstm = raw_lstm["forecast_scaled"]
        lstm = np.maximum(np.asarray(raw_lstm).reshape(-1)*bridge.scale+bridge.mean, 0)
        rows.append(pd.DataFrame({"Forecast_origin": power.index[part-4], "Target_interval_end": power.index[part],
            "Actual_kW": power["actual_kw"].iloc[part].to_numpy(), "Hybrid_kW": hybrid,
            "Task7_LSTM_kW": lstm, "Persistence_kW": power["input_kw"].iloc[part-4].to_numpy()}))
    return pd.concat(rows, ignore_index=True), {"csv_sha256": file_hash(csv_path), "source": "user-supplied later-time-range minute data",
        "scored_examples": len(ids), "origin_min": str(power.index[ids[0]-4]), "origin_max": str(power.index[ids[-1]-4]),
        "independence_limit": "Time-range separation is checked. Authenticity and absence of copied/relabelled measurements cannot be established automatically."}


def performance_tables(frame, threshold):
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, precision_score, recall_score, f1_score, confusion_matrix
    names = {"Hybrid_kW": "Integrated LSTM + ML", "Task7_LSTM_kW": "Task 7 LSTM",
             "Basic_NN_kW": "Basic neural network", "Persistence_kW": "Persistence"}
    rows = []
    for segment, mask in {"All test": np.ones(len(frame), dtype=bool), "High demand": frame["Actual_kW"].ge(threshold).to_numpy()}.items():
        y = frame.loc[mask, "Actual_kW"].to_numpy()
        for column, label in names.items():
            if column not in frame:
                continue
            p = frame.loc[mask, column].to_numpy()
            rows.append({"Model": label, "Segment": segment, "Samples": len(y),
                "MAE_kW": float(mean_absolute_error(y, p)) if len(y) else np.nan,
                "RMSE_kW": float(np.sqrt(mean_squared_error(y, p))) if len(y) else np.nan,
                "R2": float(r2_score(y, p)) if len(y) > 1 and np.var(y) > 0 else np.nan})
    y_high, p_high = frame["Actual_kW"].ge(threshold), frame["Hybrid_kW"].ge(threshold)
    tn, fp, fn, tp = confusion_matrix(y_high, p_high, labels=[False, True]).ravel()
    detection = {"Threshold_kW": threshold, "Precision": float(precision_score(y_high, p_high, zero_division=0)),
        "Recall": float(recall_score(y_high, p_high, zero_division=0)), "F1": float(f1_score(y_high, p_high, zero_division=0)),
        "True_negative": int(tn), "False_positive": int(fp), "False_negative": int(fn), "True_positive": int(tp),
        "Definition": "Thresholded hybrid power forecast, not a calibrated probability"}
    return pd.DataFrame(rows), detection


def temporal_error_profiles(frame, threshold):
    work = frame.copy()
    work["Abs_error_kW"] = (work["Actual_kW"] - work["Hybrid_kW"]).abs()
    work["Squared_error"] = (work["Actual_kW"] - work["Hybrid_kW"]) ** 2
    work["Hour"] = pd.to_datetime(work["Target_interval_end"]).dt.hour
    work["Weekend"] = pd.to_datetime(work["Target_interval_end"]).dt.dayofweek.ge(5).map({True: "Weekend", False: "Weekday"})
    work["Demand"] = np.where(work["Actual_kW"].ge(threshold), "High demand", "Other demand")
    tables = []
    for grouping in ["Hour", "Weekend", "Demand"]:
        grouped = work.groupby(grouping).agg(Samples=("Abs_error_kW", "size"), MAE_kW=("Abs_error_kW", "mean"), MSE=("Squared_error", "mean")).reset_index()
        grouped["RMSE_kW"] = np.sqrt(grouped.pop("MSE"))
        grouped = grouped.rename(columns={grouping: "Group"})
        grouped.insert(0, "Grouping", grouping)
        tables.append(grouped)
    return pd.concat(tables, ignore_index=True)


def day_block_intervals(frame, repeats=200, seed=42):
    """Uncertainty of aggregate error statistics, not per-forecast intervals."""
    work = frame.copy()
    work["Day"] = pd.to_datetime(work["Target_interval_end"]).dt.normalize()
    work["AE"] = (work["Actual_kW"] - work["Hybrid_kW"]).abs()
    work["SE"] = (work["Actual_kW"] - work["Hybrid_kW"])**2
    daily = work.groupby("Day").agg(N=("AE", "size"), AE=("AE", "sum"), SE=("SE", "sum"))
    if len(daily) < 2:
        return pd.DataFrame(), "Fewer than two days; block intervals were not calculated."
    rng = np.random.default_rng(seed)
    draws = rng.integers(0, len(daily), size=(repeats, len(daily)))
    numbers = daily["N"].to_numpy()[draws].sum(axis=1)
    mae = daily["AE"].to_numpy()[draws].sum(axis=1) / numbers
    rmse = np.sqrt(daily["SE"].to_numpy()[draws].sum(axis=1) / numbers)
    result = pd.DataFrame([{"Metric": label, "Percentile_2.5": float(np.quantile(value, .025)),
        "Percentile_97.5": float(np.quantile(value, .975)), "Day_blocks": len(daily), "Replicates": repeats}
        for label, value in [("MAE_kW", mae), ("RMSE_kW", rmse)]])
    return result, "Exploratory 95% day-block bootstrap intervals for aggregate errors; not prediction intervals. Day-to-day dependence and one-household sampling remain limitations."


def check_system_invariants(result, knowledge):
    origin = pd.Timestamp(result["forecast"]["origin"])
    target = pd.Timestamp(result["forecast"]["target_end"])
    high = result["forecast"]["forecast_kw"] >= result["forecast"]["threshold_kw"]
    scenario = Scenario(**result["scenario"])
    appliances = {a["name"]: Appliance(**a) for a in result["appliances"]}
    failures = []
    recommendations = {r["Appliance"]: r for r in result["recommendations"]}
    for name, appliance in appliances.items():
        inferred = knowledge.infer(appliance)
        row = recommendations[name]
        essential = "EssentialAppliance" in inferred["types"]
        if essential and row["Action"] != "Keep ON":
            failures.append(f"Essential appliance not protected: {name}")
        if not essential and scenario.occupancy == 0 and appliance.requires_occupancy:
            if row["Action"] not in ["Turn OFF", "Do not start"]:
                failures.append(f"Empty-home occupancy constraint violated: {name}")
        if not essential and scenario.occupancy > 0 and name == "Air Conditioner" and scenario.temperature_c >= 38:
            if row["Action"] != "Keep ON":
                failures.append("Occupied high-temperature AC comfort constraint violated")
    loads = np.zeros(scenario.horizon_hours*4, dtype=int)
    total = 0.
    for slot in result["plan"]["schedule"]:
        appliance = appliances[slot["Appliance"]]
        row = recommendations[appliance.name]
        begin, end = pd.Timestamp(slot["Start"]), pd.Timestamp(slot["End"])
        start = (begin-origin).total_seconds()/60
        finish = (end-origin).total_seconds()/60
        if start % 15 or finish % 15 or start < appliance.earliest_min or finish > min(appliance.deadline_min, scenario.horizon_hours*60) or finish-start != appliance.duration_min:
            failures.append(f"Runtime/window violation: {appliance.name}")
            continue
        if row["Kind"] != "job":
            failures.append(f"Protected/inactive appliance rescheduled: {appliance.name}")
        if not row["Flexible"] and start != appliance.earliest_min:
            failures.append(f"Non-flexible appliance delayed: {appliance.name}")
        if row["Flexible"] and high and begin < target and end > target-pd.Timedelta(minutes=15):
            failures.append(f"Flexible job overlaps single forecast high-demand interval: {appliance.name}")
        for i in range(int(start//15), int(finish//15)):
            loads[i] += 1
            total += appliance.power_w / 1000 * .25 * rate_at(origin + pd.Timedelta(minutes=15*i), scenario)
    if len(loads) and loads.max() > scenario.max_parallel_jobs:
        failures.append("Parallel-job constraint violated")
    plan = result["plan"]
    if plan["status"] in ["optimal", "feasible_search_limit"]:
        if abs(total - plan["estimated_cost"]) > 1e-7 or abs(plan["baseline_cost"]-total-plan["estimated_saving"]) > 1e-7:
            failures.append("Cost or saving calculation inconsistent")
        expected_jobs = {row["Appliance"] for row in result["recommendations"] if row["Kind"] == "job"}
        if {row["Appliance"] for row in plan["schedule"]} != expected_jobs:
            failures.append("Feasible schedule is missing requested jobs")
    elif plan["estimated_saving"] is not None or plan["estimated_cost"] is not None:
        failures.append("Unsuccessful search claimed a cost/saving")
    return failures


def scenario_suite():
    base = default_appliances()
    cases = [("Occupied household", Scenario(), base),
        ("Empty home, hot weather", Scenario(occupancy=0, temperature_c=43), base),
        ("Occupied home, very hot", Scenario(occupancy=5, temperature_c=43), base),
        ("Overnight peak tariff", Scenario(peak_start_hour=22, peak_end_hour=6), base),
        ("No peak period", Scenario(peak_start_hour=0, peak_end_hour=0), base),
        ("Two jobs permitted in parallel", Scenario(max_parallel_jobs=2), base),
        ("Short planning horizon", Scenario(horizon_hours=1), base),
        ("Flat price", Scenario(peak_rate=6, offpeak_rate=6), base)]
    fridge = Appliance("Refrigerator", 200, essential=False, flexible=True, ontology_id="Refrigerator_1")
    cases.append(("Conflicting flexible/essential ontology classes", Scenario(occupancy=0), [fridge]))
    wash = Appliance("Washing Machine", 500, ontology_id="WashingMachine_Test")
    cases.extend([("Impossible job deadline", Scenario(), [replace(wash, duration_min=60, deadline_min=30)]),
        ("Job already running", Scenario(), [replace(wash, on=True)]),
        ("User disallows shifting", Scenario(), [replace(wash, flexible=False)]),
        ("High-priority requested job", Scenario(), [replace(wash, priority="High")]),
        ("No requested inactive jobs", Scenario(), [replace(wash, requested=False)]),
        ("Very low search budget", Scenario(node_limit=1), [wash])])
    return cases


def run_scenario_evaluation(bridge, knowledge, example_count=4):
    ids = bridge._sample(bridge.test_ids, example_count)
    rows, results = [], []
    for case_id in ids:
        for name, scenario, appliances in scenario_suite():
            started = time.perf_counter()
            try:
                forecast = bridge.predict(int(case_id))
                result = optimize_decisions(forecast, appliances, scenario, knowledge)
                elapsed = time.perf_counter()-started
                failures = check_system_invariants(result, knowledge)
                # Certain cases have explicit expected outcomes, not only invariants.
                if name == "Impossible job deadline" and result["plan"]["status"] != "infeasible":
                    failures.append("Impossible deadline was not reported as infeasible")
                if name == "Very low search budget" and result["plan"]["status"] != "search_limit":
                    failures.append("Search-budget exhaustion was not reported")
                if name == "Job already running" and result["recommendations"][0]["Action"] != "Run as scheduled":
                    failures.append("Running job was interrupted")
                rows.append({"Scenario": name, "Target_index": int(case_id), "Passed": not failures,
                    "Expected_error_rejected": False, "Schedule_status": result["plan"]["status"],
                    "Latency_ms": elapsed*1000, "Violations": "; ".join(failures)})
                results.append({"scenario_name": name, "result": result})
            except Exception as error:
                rows.append({"Scenario": name, "Target_index": int(case_id), "Passed": False,
                    "Expected_error_rejected": False, "Schedule_status": "unexpected_error",
                    "Latency_ms": (time.perf_counter()-started)*1000, "Violations": repr(error)})
    forecast = bridge.predict(int(ids[0]))
    invalid = [("Negative appliance rating", [Appliance("Invalid", -100)], Scenario()),
        ("String essential flag", [Appliance("Invalid", 100, essential="false")], Scenario()),
        ("Negative tariff", [Appliance("Job", 100)], Scenario(peak_rate=-1)),
        ("Zero job concurrency", [Appliance("Job", 100)], Scenario(max_parallel_jobs=0)),
        ("Runtime not in 15-minute steps", [Appliance("Job", 100, duration_min=17)], Scenario())]
    for name, appliances, scenario in invalid:
        rejected = False
        try:
            optimize_decisions(forecast, appliances, scenario, knowledge)
        except (ValueError, TypeError):
            rejected = True
        rows.append({"Scenario": name, "Target_index": int(ids[0]), "Passed": rejected,
            "Expected_error_rejected": rejected, "Schedule_status": "expected_rejection" if rejected else "invalid_input_accepted",
            "Latency_ms": np.nan, "Violations": "" if rejected else "Invalid input accepted"})
    return pd.DataFrame(rows), results


def fairness_safety_checks(bridge, knowledge):
    forecast = bridge.predict(int(bridge.test_ids[0]))
    rows = []
    for occupancy, temp in [(0, 15), (1, 45), (8, 25)]:
        a = Appliance("Essential device A", 200, essential=True, flexible=True)
        b = replace(a, name="Essential device B")
        first = optimize_decisions(forecast, [a], Scenario(occupancy=occupancy, temperature_c=temp), knowledge)
        second = optimize_decisions(forecast, [b], Scenario(occupancy=occupancy, temperature_c=temp), knowledge)
        passed = first["recommendations"][0]["Action"] == second["recommendations"][0]["Action"] == "Keep ON"
        rows.append({"Check": "Renaming identical essential device does not change protection", "Occupancy": occupancy,
            "Temperature_C": temp, "Passed": passed, "Scope": "Controlled policy invariance; not demographic fairness"})
    return pd.DataFrame(rows)


def shap_diagnostics(bridge, frame, summary_examples=24):
    import shap
    errors = (frame["Actual_kW"]-frame["Hybrid_kW"]).abs().to_numpy()
    order = np.argsort(errors, kind="stable")
    high_rows = np.flatnonzero(frame["Actual_kW"].to_numpy() >= bridge.threshold)
    under = (frame["Actual_kW"]-frame["Hybrid_kW"]).to_numpy()
    peak = int(high_rows[np.argmax(under[high_rows])]) if len(high_rows) else int(order[-1])
    cases = {"Small error": int(order[0]), "Typical error": int(order[len(order)//2]), "High-demand diagnostic" if len(high_rows) else "Large error": peak}
    summary_rows = np.unique(np.linspace(0, len(frame)-1, min(summary_examples, len(frame)), dtype=int))
    positions = np.unique(np.r_[summary_rows, list(cases.values())])
    ids = frame["Target_index"].to_numpy(dtype=int)[positions]
    z = bridge.features(ids)
    explainer = shap.TreeExplainer(bridge.regressor)
    values = np.asarray(explainer.shap_values(z))
    if values.shape == z.shape + (1,):
        values = values[..., 0]
    if values.shape != z.shape or not np.isfinite(values).all():
        raise ValueError("Unexpected or non-finite SHAP values.")
    base = float(np.asarray(explainer.expected_value).reshape(-1)[0])
    raw = np.asarray(bridge.regressor.predict(z))
    residual = raw - base - values.sum(axis=1)
    tolerance = 1e-5 + 1e-5*np.abs(raw)
    checks = pd.DataFrame({"Frame_position": positions, "Target_index": ids, "Raw_forecast_kW": raw,
        "Clipped_forecast_kW": np.maximum(raw, 0), "SHAP_base_kW": base, "SHAP_sum_kW": values.sum(axis=1),
        "Residual_kW": residual, "Tolerance_kW": tolerance, "Passed": np.abs(residual) <= tolerance})
    lookup = {int(pos): row for row, pos in enumerate(positions)}
    summary_index = [lookup[int(pos)] for pos in summary_rows]
    importance = pd.DataFrame({"Feature": bridge.feature_names, "Mean_abs_SHAP_kW": np.abs(values[summary_index]).mean(axis=0)}).sort_values("Mean_abs_SHAP_kW", ascending=False)
    return {"positions": positions, "cases": cases, "lookup": lookup, "values": values, "data": z,
        "base": base, "checks": checks, "importance": importance, "summary_examples": len(summary_rows)}


def responsible_ai_table():
    return pd.DataFrame([
        ("Generalization and bias", "One UCI household and a historical period do not represent all homes or regions.", "Report time/demand error groups; validate on other households before wider use.", "No population or demographic fairness claim."),
        ("Comfort and accessibility", "A fixed 38°C AC rule may not fit individual health or comfort needs.", "Protect essential devices; require user review and future personalizable thresholds.", "Not a medical or personalized comfort guarantee."),
        ("False peak alerts and missed peaks", "Forecast errors can delay jobs unnecessarily or miss high demand.", "Report threshold confusion counts and keep essential/comfort overrides.", "No calibrated confidence probability or guaranteed savings."),
        ("Privacy", "Fine-grained electricity data may reveal household routines.", "Minimize retention; remove identifiers; control sharing of the runtime, notebook and exports.", "Colab is cloud-hosted; this is not a local-only privacy guarantee."),
        ("User control and safety", "Automatic switching could disrupt important appliances or running cycles.", "Prototype gives recommendations only; preserve essential/running jobs and report infeasibility.", "No hardware-control or production-safety certification."),
        ("Tariffs and savings", "Example ₹ tariffs, ratings and durations may not match an actual bill.", "Label scenario inputs and estimate only requested-job savings against the stated feasible baseline.", "Do not claim measured whole-house bill reductions."),
        ("Explainability", "Correlated latent/context features can share SHAP credit; activations are not causal importance.", "Name the explained ML stage, verify reconstruction, and show rule traces separately.", "Additivity is not proof of causal validity or complete reliability."),
        ("Legal and data governance", "Permission, data-source terms and applicable data obligations depend on deployment context.", "Document source/provenance and obtain appropriate permission before using identifiable household data.", "No jurisdiction-specific compliance determination was performed.")],
        columns=["Topic", "Risk", "Mitigation", "Residual_limit"])


def write_evaluation_documents(output_dir, bridge, metrics, scenarios, fairness, xai, freshness, intervals_note, setup_mode, fingerprint):
    output_dir = Path(output_dir)
    overall = metrics[metrics["Segment"].eq("All test")].set_index("Model")
    hybrid = overall.loc["Integrated LSTM + ML"]
    lstm = overall.loc["Task 7 LSTM"]
    passed, total = int(scenarios["Passed"].sum()), len(scenarios)
    shap_passed, shap_total = int(xai["checks"]["Passed"].sum()), len(xai["checks"])
    performance_direction = "lower" if hybrid["RMSE_kW"] < lstm["RMSE_kW"] else "higher or equal"
    report = f"""# Task 10 — final evaluation

## Objective and system architecture
Real recorded electricity history → Task 7 causal preparation → frozen LSTM patterns
→ validation-selected classical ML prediction → imported ontology/SWRL and operational
rules → constrained A* job scheduling → dashboard and explanations.

## Method and data provenance
Task 7 run: {bridge.manifest['run_id']}. ML stage: {bridge.selected_name}.
Setup mode: {setup_mode}. Models were fingerprinted before evaluation and checked
afterward. Model preparation, if required by a fresh runtime, preceded the freeze;
no evaluation results were used to fit/tune models.

Existing test observations are disjoint from training and validation, but this
same held-out set was already evaluated in Task 9. They are unseen to fitting,
not a new independent test sample. Additional household scenarios are controlled
designed combinations, not measured new households or proof of population fairness.
Independent later-time-range CSV supplied: {freshness['external_csv_supplied']}.
See data_freshness.json and any external evaluation files for exact coverage.

## Forecast results
Integrated model: MAE {hybrid['MAE_kW']:.4f} kW, RMSE {hybrid['RMSE_kW']:.4f} kW,
R² {hybrid['R2']:.4f}, evaluated on {int(hybrid['Samples'])} test forecasts.
Task 7 LSTM RMSE: {lstm['RMSE_kW']:.4f} kW. The integrated model has
{performance_direction} RMSE on this period. Integration is not automatically
an accuracy improvement. Demand/hour/weekday profiles and high-demand metrics
are saved separately; subgroup R² is not directly comparable with overall R².
{intervals_note}

## End-to-end correctness and runtime
{passed}/{total} scenario/error-handling checks passed. Outcomes include feasible,
infeasible and search-limit cases: successfully reporting infeasibility is not
counted as a system failure when expected. Latencies exclude SHAP and plotting,
include inference and rule/scheduling execution, and are specific to this hardware.
Inspect all failed rows rather than hiding them. Controlled policy-invariance
checks passed: {int(fairness['Passed'].sum())}/{len(fairness)}.

## Explainability
TreeSHAP reconstructs the raw ML-stage prediction in kW: {shap_passed}/{shap_total}
checks passed. Final negative-value clipping is reported separately. The features
include learned LSTM patterns and context; this is not an exact sensor-level
explanation of the whole composed system. Diagnostic cases include a small-error,
typical-error and high-demand/large-error example; they are deliberately selected,
not an unbiased accuracy sample. LSTM activations inspect internal responses;
rule traces explain operational actions. Neither method establishes causality.

## Responsible AI and deployment
See responsible_ai.csv/model_card.md for privacy, essential-appliance protection,
comfort limitations, user control, tariff assumptions, representativeness and data
governance. No demographic labels are available; temporal groups and controlled
device-renaming checks do not establish demographic fairness. ₹ settings and
savings are scenario assumptions; they are not actual French-household bills.
The deployment is a Colab notebook prototype, not a continuously hosted service
or hardware controller. The supplied OWL/SWRL adapter is a documented supported
subset, not a general OWL DL consistency reasoner.

## Remaining limitations
The previously examined test set limits claims of independent validation. More
households and fresh measured future inputs are needed for broad generalization.
Task 10 does not guarantee accuracy, production safety, regulatory compliance or
realized savings. Failed scenario/XAI checks, if any, remain visible in exports.
"""
    (output_dir / "evaluation_report.md").write_text(report, encoding="utf-8")
    card = f"""# Model/system card
Intended use: academic household demand prediction and advisory job scheduling.
Not intended for autonomous appliance switching, medical decisions or guaranteed bills.
Data: UCI Individual Household Electric Power Consumption; one household, 2006–2010.
Inputs: 96 past 15-minute multivariate steps plus frozen context features; separate
household settings enter rules/scheduling, not the learned forecast.
Target: mean kW in the 15-minute interval ending one hour after forecast origin.
Model: frozen Task 7 LSTM + {bridge.selected_name}; Task 9 selects ML on validation only.
Evaluation: reused chronological test period, additional designed scenarios,
optional externally supplied later-time-range data. External CSV supplied: {freshness['external_csv_supplied']}.
Uncertainty: error-statistic block intervals are exploratory; no calibrated prediction intervals.
Fairness: demographic data absent; no demographic fairness certification.
Human oversight: users review/reject recommendations; essential and running jobs protected.
Data handling: runtime/exported files need access controls and retention decisions;
Colab is cloud-hosted. No external household-data API is invoked during evaluation.
Explanations: ML-head TreeSHAP, LSTM activations, explicit rule and scheduling traces.
"""
    (output_dir / "model_card.md").write_text(card, encoding="utf-8")
    notes = f"""# Structured presentation notes
1. **Problem and scope:** anticipate demand and suggest appliance schedules; advisory prototype.
2. **Architecture:** causal data prep → LSTM representation → classical ML → ontology/rules → A* → dashboard.
3. **Method:** chronological partitions, training-only transformations, validation selection, frozen evaluation.
4. **Results:** MAE {hybrid['MAE_kW']:.4f} kW; RMSE {hybrid['RMSE_kW']:.4f} kW; R² {hybrid['R2']:.4f}; compare LSTM and persistence honestly.
5. **Working demonstration:** change a historical case and household settings; show schedule, cost assumptions and rule trace.
6. **Explanation example:** display an error case, TreeSHAP reconstruction and LSTM activations; distinguish attribution from causality.
7. **Validation/responsible AI:** {passed}/{total} designed checks; essential/comfort protection, privacy, representativeness and unavailable demographic fairness evidence.
8. **Limits and future work:** reused holdout, new-household/future measurements, personalization and external validation before wider deployment.
"""
    (output_dir / "presentation_notes.md").write_text(notes, encoding="utf-8")
    guide = """# Deployment and demonstration guide
Open the standalone Task 10 notebook in Colab and run its cells. Completed Task 7
folders and the Task 9 hybrid cache are reused when accessible. An explicit saved
folder can be configured. A separate Colab runtime does not automatically share
temporary files; if permitted, initial preparation rebuilds Task 9 once and labels
the run as rebuilt. Preparation finishes before models are frozen for evaluation.
Use the optional final dashboard for an interactive demonstration. Changing its
settings performs inference/scheduling, not training. Export and retain the executed
notebook and evaluation folder. Sharing a notebook is not a continuously available
production deployment. Keep private raw household data and identifiers out of
public submissions. No hardware is controlled.
"""
    (output_dir / "deployment_guide.md").write_text(guide, encoding="utf-8")
    (output_dir / "frozen_model_fingerprint.json").write_text(json.dumps(fingerprint, indent=2), encoding="utf-8")
    return report


TASK7_PREPROCESSORS = {"np": np, "pd": pd, "ELECTRICAL_FEATURES": ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3'], "TARGET_FEATURE": "Global_active_power", "HISTORY_INPUT_COLUMNS": ['input_Global_active_power', 'input_Global_reactive_power', 'input_Voltage', 'input_Global_intensity', 'input_Sub_metering_1', 'input_Sub_metering_2', 'input_Sub_metering_3']}
exec(compile(TASK7_PREPROCESSOR_SOURCE, "frozen_task7_preprocessors.py", "exec"), TASK7_PREPROCESSORS)

## 3. Prepare the system, then freeze it
Preparation may reuse cached models or, if permitted and necessary, rebuild
them using the unchanged Task 9 training procedure. The following evaluation
uses fixed model weights, a fixed ML head and training-derived threshold.

In [ ]:
def load_evaluation_system():
    from unittest.mock import patch
    from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
    if LIVE_TASK9_BRIDGE is not None and all(hasattr(LIVE_TASK9_BRIDGE, n) for n in ["network", "regressor", "test_ids", "signature", "folder"]):
        if not TASK7_FOLDER or Path(TASK7_FOLDER).resolve() == LIVE_TASK9_BRIDGE.folder.resolve():
            return LIVE_TASK9_BRIDGE, "reused_live_task9_models"
    roots = [Path.cwd()]
    for key in ["TASK7_COMPLETED_OUTPUT_DIR", "ARTIFACT_DIR"]:
        if globals().get(key):
            roots.append(Path(globals()[key]))
    folder = find_task7_folder(TASK7_FOLDER, roots)
    rebuilt_task7 = folder is None
    if folder is None:
        if not PREPARE_MODELS_IF_MISSING:
            raise FileNotFoundError("No completed Task 7 model folder is available; preparation is disabled.")
        folder = prepare_task7_results()
    cache_before = {str(path.resolve()) for path in (TASK9_OUTPUT_DIR / "model_cache").glob("*/hybrid_forecaster.joblib")}
    if PREPARE_MODELS_IF_MISSING:
        current = ForecastBridge(folder, TASK9_OUTPUT_DIR, TRAIN_EXAMPLES, VALIDATION_EXAMPLES, False)
    else:
        message = "No matching frozen Task 9 ML cache. Supply the original cache or enable model preparation before evaluation."
        with patch.object(GradientBoostingRegressor, "fit", side_effect=RuntimeError(message)), patch.object(RandomForestRegressor, "fit", side_effect=RuntimeError(message)):
            current = ForecastBridge(folder, TASK9_OUTPUT_DIR, TRAIN_EXAMPLES, VALIDATION_EXAMPLES, False)
    cache_file = str((current.cache / "hybrid_forecaster.joblib").resolve())
    mode = "rebuilt_task7_and_task9_before_freeze" if rebuilt_task7 else "prepared_missing_task9_ml_before_freeze" if cache_file not in cache_before else "reused_saved_task9_models"
    return current, mode

bridge, setup_mode = load_evaluation_system()
knowledge = OntologyKnowledge(ONTOLOGY_XML)
frozen_before = freeze_fingerprint(bridge, knowledge)
print("Setup:", setup_mode, "| Task 7 run:", bridge.manifest["run_id"], "| Model:", bridge.selected_name)
print("EVALUATION STARTS HERE: model weights and ML predictions are now frozen.")
print("Output folder:", OUTPUT_DIR_10.resolve())

## 4. Forecast performance on inputs excluded from fitting
All saved test examples are disjoint from training and validation; temporal
boundaries and target alignment are checked. This is the same holdout already
examined in Task 9. MAE/RMSE are lower-is-better; R² is higher-is-better.
High-demand subgroup R² can be negative and is not directly comparable with
the overall R² because the target variance differs.

Day-block bootstrap intervals describe uncertainty of aggregate error statistics,
not the uncertainty of a particular prediction. One household and day-to-day
dependence limit these exploratory intervals.

In [ ]:
evaluation_frame = collect_frozen_predictions(bridge)
evaluation_frame.to_csv(OUTPUT_DIR_10 / "reused_holdout_predictions.csv", index=False)
metrics, peak_detection = performance_tables(evaluation_frame, bridge.threshold)
groups = temporal_error_profiles(evaluation_frame, bridge.threshold)
intervals, intervals_note = day_block_intervals(evaluation_frame, BOOTSTRAP_REPLICATES)
metrics.to_csv(OUTPUT_DIR_10 / "forecast_performance.csv", index=False)
groups.to_csv(OUTPUT_DIR_10 / "temporal_demand_error_profiles.csv", index=False)
intervals.to_csv(OUTPUT_DIR_10 / "error_statistic_block_intervals.csv", index=False)
(OUTPUT_DIR_10 / "peak_detection.json").write_text(json.dumps(peak_detection, indent=2), encoding="utf-8")
display(metrics.round(4))
display(intervals.round(4))
print(intervals_note)
print("Thresholded hybrid peak detection:", peak_detection)
freshness = {"external_csv_supplied": bool(UNSEEN_MINUTE_CSV), "test_disjoint_from_fitting": True,
    "existing_test_already_evaluated_in_task9": True, "existing_test_claimed_independent_new_test": False,
    "household_scenarios": "Additional designed combinations, not measured new-household observations",
    "test_examples": len(evaluation_frame), "model_preparation_mode": setup_mode}

## 5. Optional independent later-time-range observations
Leave the path empty if none are available. The report then explicitly records
that no independent new measured dataset was supplied. The CSV must contain
**Datetime** (ISO local minute timestamps without timezone suffix) and:
`Global_active_power`, `Global_reactive_power`, `Voltage`, `Global_intensity`,
`Sub_metering_1`, `Sub_metering_2`, `Sub_metering_3`.

Supply at least eight days of minute readings, including warm-up for the 7-day
context lag. Only complete observed target intervals with forecast origins later
than the original dataset end are scored. Original training-only scalers and
Task 7 causal transformations are applied without fitting or interpolation from
future readings. Date separation alone cannot establish data authenticity or
guarantee independent households. Missing columns or insufficient history fail
clearly rather than generating invented readings.

In [ ]:
external_metrics = None
if UNSEEN_MINUTE_CSV:
    external_frame, external_provenance = external_csv_predictions(bridge, UNSEEN_MINUTE_CSV, TASK7_PREPROCESSORS)
    external_metrics, external_detection = performance_tables(external_frame, bridge.threshold)
    external_frame.to_csv(OUTPUT_DIR_10 / "independent_csv_predictions.csv", index=False)
    external_metrics.to_csv(OUTPUT_DIR_10 / "independent_csv_performance.csv", index=False)
    freshness["external_provenance"] = external_provenance
    display(external_metrics.round(4))
else:
    print("No independent external measurements supplied. Existing holdout results are labelled as reused.")
(OUTPUT_DIR_10 / "data_freshness.json").write_text(json.dumps(freshness, indent=2), encoding="utf-8")

In [ ]:
"""Task 10 publication-style evaluation plots; saved alongside the report."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML


def save_eval_figure(output_dir, name):
    plt.gcf().savefig(output_dir / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")


def plot_evaluation(frame, metrics, groups, detection, output_dir):
    overall = metrics[metrics["Segment"].eq("All test")].set_index("Model")
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    for ax, column in zip(axes, ["MAE_kW", "RMSE_kW", "R2"]):
        bars = ax.bar(overall.index, overall[column], color=["#176b87", "#31a6a0", "#7da2bc", "#a7b0bc"][:len(overall)])
        ax.bar_label(bars, fmt="%.3f", fontsize=8, padding=3)
        ax.set_title(column.replace("_", " "))
        ax.tick_params(axis="x", rotation=35, labelsize=8)
        ax.axhline(0, color="black", linewidth=.6)
        ax.margins(y=.2)
    fig.suptitle("Frozen system: previously evaluated holdout, reused transparently")
    fig.tight_layout()
    save_eval_figure(output_dir, "forecast_comparison")
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    hourly = groups[groups["Grouping"].eq("Hour")]
    axes[0].bar(hourly["Group"].astype(int), hourly["MAE_kW"], color="#176b87")
    axes[0].set(title="Time-of-day error profile", xlabel="Target interval end hour", ylabel="MAE (kW)")
    weekend = groups[groups["Grouping"].eq("Weekend")]
    axes[1].bar(weekend["Group"].astype(str), weekend["MAE_kW"], color="#31a6a0")
    axes[1].set(title="Weekday/weekend error profile", ylabel="MAE (kW)")
    confusion = np.array([[detection["True_negative"], detection["False_positive"]],
                          [detection["False_negative"], detection["True_positive"]]])
    image = axes[2].imshow(confusion, cmap="Blues")
    for i in range(2):
        for j in range(2):
            axes[2].text(j, i, str(confusion[i, j]), ha="center", va="center",
                         color="white" if confusion[i, j] > confusion.max()/2 else "black")
    axes[2].set_xticks([0, 1], ["Normal", "High"])
    axes[2].set_yticks([0, 1], ["Normal", "High"])
    axes[2].set(title="Thresholded hybrid forecast", xlabel="Predicted demand", ylabel="Actual demand")
    fig.colorbar(image, ax=axes[2], label="Examples")
    fig.tight_layout()
    save_eval_figure(output_dir, "error_profiles_and_peak_detection")
    fig, ax = plt.subplots(figsize=(10, 4))
    residual = frame["Actual_kW"]-frame["Hybrid_kW"]
    ax.hist(residual, bins=45, color="#176b87", alpha=.85)
    ax.axvline(0, color="black", linewidth=.8)
    ax.set(title="Hybrid forecast errors: positive = underprediction", xlabel="Actual − predicted (kW)", ylabel="Count")
    fig.tight_layout()
    save_eval_figure(output_dir, "forecast_error_distribution")


def plot_scenario_checks(checks, output_dir):
    valid_times = checks["Latency_ms"].dropna()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    counts = [int(checks["Passed"].sum()), int((~checks["Passed"]).sum())]
    axes[0].bar(["Passed", "Failed"], counts, color=["#31a6a0", "#ce5757"])
    axes[0].set(title="Designed scenario and invalid-input checks", ylabel="Check count")
    if len(valid_times):
        axes[1].hist(valid_times, bins=min(20, max(3, len(valid_times)//2)), color="#176b87")
    axes[1].set(title="Inference + reasoning + scheduling latency", xlabel="Milliseconds (SHAP/rendering excluded)", ylabel="Count")
    fig.tight_layout()
    save_eval_figure(output_dir, "system_checks_and_latency")


def plot_explanations(bridge, frame, xai, output_dir):
    import shap
    importance = xai["importance"].head(15).iloc[::-1]
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.barh(importance["Feature"], importance["Mean_abs_SHAP_kW"], color="#176b87")
    ax.set(xlabel="Mean absolute SHAP contribution (kW)", title=f"ML-stage importance over {xai['summary_examples']} evenly spaced test examples")
    fig.tight_layout()
    save_eval_figure(output_dir, "shap_global_importance")
    case_rows = []
    for label, position in xai["cases"].items():
        row = xai["lookup"][position]
        actual = frame.iloc[position]
        obj = shap.Explanation(values=xai["values"][row], base_values=xai["base"], feature_names=bridge.feature_names)
        shap.plots.waterfall(obj, max_display=12, show=False)
        plt.gcf().set_size_inches(12, 7)
        plt.title(f"{label}: actual {actual['Actual_kW']:.3f} kW; final forecast {actual['Hybrid_kW']:.3f} kW\n"
                  f"SHAP explains raw ML output; reconstruction residual {xai['checks'].iloc[row]['Residual_kW']:+.6f} kW", fontsize=10)
        plt.tight_layout()
        safe_label = label.lower().replace(" ", "_").replace("-", "_")
        save_eval_figure(output_dir, "shap_" + safe_label)
        activation = bridge.activation(int(actual["Target_index"]))
        fig, ax = plt.subplots(figsize=(12, 4))
        image = ax.imshow(activation.T, aspect="auto", cmap="RdBu_r")
        ticks = np.linspace(0, activation.shape[0]-1, 7, dtype=int)
        ax.set_xticks(ticks, [f"{(activation.shape[0]-1-t)*.25:g}" for t in ticks])
        ax.set(title=label + ": LSTM activations (not attention weights)", xlabel="Hours before forecast origin", ylabel="LSTM unit")
        fig.colorbar(image, ax=ax, label="Hidden-state activation")
        fig.tight_layout()
        save_eval_figure(output_dir, "activations_" + safe_label)
        case_rows.append({"Case": label, "Target_index": int(actual["Target_index"]), "Forecast_origin": str(actual["Forecast_origin"]),
                          "Target_interval_end": str(actual["Target_interval_end"]), "Actual_kW": actual["Actual_kW"],
                          "Predicted_kW": actual["Hybrid_kW"], "Absolute_error_kW": abs(actual["Actual_kW"]-actual["Hybrid_kW"])})
    return pd.DataFrame(case_rows)

In [ ]:
plot_evaluation(evaluation_frame, metrics, groups, peak_detection, OUTPUT_DIR_10)
display(groups.round(4))
print("Hour/weekend/demand groups are performance diagnostics, not demographic fairness evidence.")

## 6. End-to-end scenario testing
Each scenario uses a real model forecast and user-specified household settings.
Checks cover ontology protection, occupied/empty hot homes, overnight and flat
tariffs, concurrency, short windows, running jobs, user restrictions, impossible
deadlines and search limits. Invalid-input cases must be rejected clearly.
Checks independently reconstruct costs and verify schedule constraints.
Successfully reporting infeasibility is a correct outcome when constraints
cannot be met. Latency includes inference, reasoning and scheduling; excludes
SHAP and rendering and does not imply production latency guarantees.

In [ ]:
scenario_checks, scenario_results = run_scenario_evaluation(bridge, knowledge, SCENARIO_HISTORY_EXAMPLES)
scenario_checks.to_csv(OUTPUT_DIR_10 / "end_to_end_scenario_checks.csv", index=False)
(OUTPUT_DIR_10 / "scenario_inputs_and_results.json").write_text(json.dumps(scenario_results, indent=2), encoding="utf-8")
display(scenario_checks)
latencies = scenario_checks["Latency_ms"].dropna()
latency_summary = {"Timed_checks": len(latencies), "Median_ms": float(latencies.median()),
    "P95_ms": float(latencies.quantile(.95)), "First_timed_call_ms": float(latencies.iloc[0]),
    "Definition": "Includes inference + rules + scheduling; excludes explanation/rendering; first call may include warm-up"}
(OUTPUT_DIR_10 / "latency_summary.json").write_text(json.dumps(latency_summary, indent=2), encoding="utf-8")
print("Scenario checks passed:", int(scenario_checks["Passed"].sum()), "/", len(scenario_checks))
print("Latency:", latency_summary)
plot_scenario_checks(scenario_checks, OUTPUT_DIR_10)

## 7. Explain predictions and operational decisions
TreeSHAP explains the ML head using learned LSTM patterns and context inputs.
It does not provide exact sensor-level attributions for the whole composed
system. The tree reference is based on fitted training-path statistics.
Reconstruction checks concern the raw ML output; clipping to non-negative
power is shown separately. Passing a reconstruction check does not prove causal
validity or general reliability. LSTM activation maps are internal responses,
not attention weights or causal feature importance.

Summary examples are evenly spaced. Small-error, typical-error and high-demand/
large-error cases are deliberately selected diagnostics; they are not an unbiased
performance sample. Rule traces separately explain why an appliance action was
selected and how ontology candidates were resolved.

In [ ]:
xai = shap_diagnostics(bridge, evaluation_frame, SHAP_SUMMARY_EXAMPLES)
xai["checks"].to_csv(OUTPUT_DIR_10 / "shap_reconstruction_checks.csv", index=False)
xai["importance"].to_csv(OUTPUT_DIR_10 / "shap_feature_importance.csv", index=False)
np.savez_compressed(OUTPUT_DIR_10 / "ml_shap_values.npz", values=xai["values"], features=np.array(bridge.feature_names),
    base_value=xai["base"], test_frame_positions=xai["positions"])
diagnostics = plot_explanations(bridge, evaluation_frame, xai, OUTPUT_DIR_10)
diagnostics.to_csv(OUTPUT_DIR_10 / "diagnostic_forecasts.csv", index=False)
display(diagnostics.round(4))
display(xai["checks"].round(6))
rule_examples = []
for label, position in xai["cases"].items():
    forecast = bridge.predict(int(evaluation_frame.iloc[position]["Target_index"]))
    decision = optimize_decisions(forecast, default_appliances(), Scenario(), knowledge)
    for row in decision["recommendations"]:
        rule_examples.append({"Case": label, "Appliance": row["Appliance"], "Action": row["Action"],
            "Rule_trace": " → ".join(row["Rule_trace"]), "Reason": row["Reason"],
            "Suggested_start": row.get("Suggested_start"), "Schedule_status": decision["plan"]["status"]})
rule_examples = pd.DataFrame(rule_examples)
rule_examples.to_csv(OUTPUT_DIR_10 / "explained_decision_paths.csv", index=False)
display(rule_examples)

## 8. Ethics, fairness, privacy, safety and bias
The source dataset contains no demographic labels. We therefore do not invent
gender/income/ethnicity fairness scores. Temporal error groups describe error
distribution, while controlled checks verify limited policy invariances.
Neither establishes population fairness. Essential-device protection is tested
under renaming and different occupancy/temperature inputs.

Practical limits include one-household training, fixed comfort thresholds,
incorrect peak forecasts, scenario tariff assumptions, privacy risks and the
advisory nature of this prototype. Model interpretation is not a causal proof.
Data permissions/terms and legal obligations depend on deployment context; no
jurisdiction-specific compliance certification is claimed.

In [ ]:
fairness_checks = fairness_safety_checks(bridge, knowledge)
responsible_ai = responsible_ai_table()
fairness_checks.to_csv(OUTPUT_DIR_10 / "controlled_policy_invariance.csv", index=False)
responsible_ai.to_csv(OUTPUT_DIR_10 / "responsible_ai.csv", index=False)
display(fairness_checks)
display(responsible_ai)

## 9. Working prototype demonstration
The optional dashboard reuses the exact frozen Task 9 bridge. It allows a
demonstrator to change household settings and inspect recommendations. A
Colab notebook is a prototype deployment; saving/sharing it is not a continuously
hosted production service. No hardware is controlled. Keep private inputs and
identifiers out of public exports. The same dashboard is available in Task 9.

In [ ]:
"""Interactive in-notebook controls and Task 9 result views."""
from datetime import datetime, timezone
from html import escape
import json
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, clear_output, display

if "Appliance" not in globals():
    from task9_core import Appliance, Scenario, default_appliances, optimize_decisions


def render_optimizer_result(result, bridge, explanation=None):
    forecast, plan = result["forecast"], result["plan"]
    high = forecast["forecast_kw"] >= forecast["threshold_kw"]
    saving = "Unavailable" if plan["estimated_saving"] is None else f"₹{plan['estimated_saving']:.2f}"
    cards = [("FORECAST AVERAGE POWER", f"{forecast['forecast_kw']:.3f} kW"),
             ("FORECAST DEMAND", "High" if high else "Normal"),
             ("SCHEDULE STATUS", plan["status"].replace("_", " ")),
             ("ESTIMATED JOB SAVING", saving)]
    markup = '<div style="display:flex;flex-wrap:wrap;gap:12px;font-family:Arial,sans-serif">'
    for label, value in cards:
        markup += f'<div style="background:#edf5fa;border-left:4px solid #176b87;padding:15px;min-width:185px;border-radius:8px"><small>{escape(label)}</small><br><strong style="font-size:23px">{escape(value)}</strong></div>'
    markup += '</div>'
    display(HTML(markup))
    display(HTML(f"<p><b>Origin:</b> {escape(forecast['origin'])} &nbsp; <b>Predicted interval:</b> "
                 f"{escape(forecast['target_interval_start'])} to {escape(forecast['target_end'])}<br>"
                 f"<b>Model:</b> {escape(forecast['model'])}. High demand starts at {forecast['threshold_kw']:.3f} kW.</p>"))
    display(HTML("<p>Historical electricity case with editable household scenario settings. "
                 "Savings cover the requested schedulable jobs versus a feasible earliest-start schedule; they are estimates, not a measured household bill.</p>"))
    rows = [{"Appliance": r["Appliance"], "Recommendation": r["Action"],
             "Suggested start": r.get("Suggested_start") or "Protected / inactive",
             "Reason": r["Reason"]} for r in result["recommendations"]]
    display(pd.DataFrame(rows))
    display(HTML(f"<p><b>Search:</b> {escape(plan['reason'])} Explored {plan['nodes']} A* states. "
                 "The concurrency setting counts scheduled jobs, not the entire household.</p>"))
    input_values = bridge.inputs(np.array([forecast["target_index"]]))
    history_names = bridge.manifest["history_features"]
    power_channel = next(i for i, name in enumerate(history_names) if "global_active_power" in name.lower())
    history_kw = input_values["history"][0, :, power_channel] * bridge.scale + bridge.mean
    origin, target = pd.Timestamp(forecast["origin"]), pd.Timestamp(forecast["target_end"])
    history_times = pd.date_range(end=origin, periods=len(history_kw), freq="15min")
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].plot(history_times, history_kw, color="#176b87", label="Historical model inputs")
    axes[0].scatter([target], [forecast["forecast_kw"]], color="#d86526", s=65, label="One-hour-ahead target")
    axes[0].axvspan(target-pd.Timedelta(minutes=15), target, alpha=.15, color="#d86526")
    axes[0].set(title="Recent history and forecast target", ylabel="Average power (kW)")
    axes[0].legend(fontsize=8)
    axes[0].tick_params(axis="x", rotation=25)
    if plan["estimated_cost"] is not None:
        axes[1].bar(["Feasible earliest start", "A* schedule"], [plan["baseline_cost"], plan["estimated_cost"]], color=["#94a3b8", "#31a6a0"])
        axes[1].set(title="Estimated cost of scheduled jobs", ylabel="Scenario cost (₹)")
    else:
        axes[1].text(.5, .5, "No feasible cost estimate\nReview scheduling constraints", ha="center", va="center", transform=axes[1].transAxes)
    fig.tight_layout()
    plt.show()
    plt.close(fig)
    if plan["schedule"]:
        schedule = pd.DataFrame(plan["schedule"])
        display(schedule)
        fig, ax = plt.subplots(figsize=(12, 3.5))
        for i, row in schedule.iterrows():
            start = (pd.Timestamp(row["Start"]) - origin).total_seconds() / 3600
            duration = (pd.Timestamp(row["End"]) - pd.Timestamp(row["Start"])).total_seconds() / 3600
            ax.barh(i, duration, left=start, color="#31a6a0", height=.55)
        ax.set_yticks(range(len(schedule)), schedule["Appliance"])
        tick_values = np.arange(result["scenario"]["horizon_hours"] + 1)
        ax.set_xticks(tick_values, [(origin + pd.Timedelta(hours=int(h))).strftime("%H:%M") for h in tick_values])
        if high:
            ax.axvspan(.75, 1., alpha=.15, color="red", label="Single forecast high-demand interval")
            ax.legend(fontsize=8)
        ax.set(xlabel=f"Planning clock from {origin.date()}", title="Requested-job schedule")
        fig.tight_layout()
        plt.show()
        plt.close(fig)
    with_details = "<details><summary><b>Why were these decisions made? — ontology and operational rule trace</b></summary>"
    for row in result["recommendations"]:
        with_details += f"<p><b>{escape(row['Appliance'])}</b>: {escape(' → '.join(row['Rule_trace']))}<br>"
        with_details += f"OWL candidate decisions: {escape(', '.join(row['OWL_candidates']) or 'None')}. "
        with_details += f"Final decision: {escape(row['Action'])}. {escape(row['Reason'])}</p>"
    display(HTML(with_details + "</details>"))
    if explanation is not None:
        import shap
        display(HTML("<h4>SHAP: why the ML prediction stage produced this forecast</h4>"
                     "<p>Features include learned LSTM patterns and context inputs. These SHAP values explain the ML stage, "
                     "not separate causal effects of raw sensors. LSTM activations are shown below.</p>"))
        obj = shap.Explanation(values=explanation["values"], base_values=explanation["base"],
                               feature_names=explanation["features"])
        shap.plots.waterfall(obj, max_display=12, show=False)
        fig = plt.gcf()
        fig.set_size_inches(12, 7)
        plt.title(f"Raw ML forecast: {explanation['raw_prediction']:.3f} kW; final clipped forecast: {forecast['forecast_kw']:.3f} kW", fontsize=10)
        fig.tight_layout()
        plt.show()
        plt.close(fig)
        activation = bridge.activation(forecast["target_index"])
        fig, ax = plt.subplots(figsize=(12, 4))
        image = ax.imshow(activation.T, aspect="auto", cmap="RdBu_r")
        ticks = np.linspace(0, activation.shape[0]-1, 7, dtype=int)
        ax.set_xticks(ticks, [f"{(activation.shape[0]-1-t)*.25:g}" for t in ticks])
        ax.set(title="LSTM temporal activations — internal response, not attention weights", xlabel="Hours before forecast origin", ylabel="LSTM unit")
        fig.colorbar(image, ax=ax, label="Hidden-state activation")
        fig.tight_layout()
        plt.show()
        plt.close(fig)


def build_optimizer_dashboard(bridge, knowledge, output_dir, performance=None):
    from dataclasses import asdict
    # Defined in task9_core, available as imports locally or as earlier notebook definitions.
    ids = bridge._sample(bridge.test_ids, 64)
    evening = bridge.test_ids[bridge.timestamps[bridge.test_ids - 4].hour == 18]
    if len(evening):
        ids = np.unique(np.r_[ids, evening[0]])
    case = widgets.Dropdown(options=[(str(bridge.timestamps[int(i)-4]), int(i)) for i in ids],
                            value=int(evening[0] if len(evening) else ids[0]), description="History case:", layout=widgets.Layout(width="470px"))
    occupancy = widgets.IntSlider(value=3, min=0, max=10, description="People home:")
    temperature = widgets.FloatSlider(value=35, min=0, max=50, step=.5, description="Temp (°C):")
    peak_rate = widgets.BoundedFloatText(value=9, min=0, max=1000, description="Peak ₹/kWh:")
    offpeak_rate = widgets.BoundedFloatText(value=6, min=0, max=1000, description="Other ₹/kWh:")
    peak_start = widgets.IntSlider(value=18, min=0, max=23, description="Peak starts:")
    peak_end = widgets.IntSlider(value=20, min=0, max=23, description="Peak ends:")
    horizon = widgets.Dropdown(options=[4, 6, 8, 12], value=6, description="Plan hours:")
    concurrency = widgets.IntSlider(value=1, min=1, max=3, description="Parallel jobs:")
    appliance_forms, appliance_sections = [], []
    for appliance in default_appliances():
        initial = asdict(appliance)
        locked = "EssentialAppliance" in knowledge.infer(appliance)["types"]
        options = {"requested": widgets.Checkbox(value=appliance.requested, description="Requested"),
            "on": widgets.Checkbox(value=appliance.on, description="Already running"),
            "essential": widgets.Checkbox(value=appliance.essential or locked, description="Essential", disabled=locked),
            "flexible": widgets.Checkbox(value=appliance.flexible, description="Allow rescheduling"),
            "requires_occupancy": widgets.Checkbox(value=appliance.requires_occupancy, description="Needs people home"),
            "priority": widgets.Dropdown(value=appliance.priority, options=["Low", "Medium", "High"], description="Priority:"),
            "power_w": widgets.BoundedFloatText(value=appliance.power_w, min=1, max=20000, description="Power (W):"),
            "duration_min": widgets.BoundedIntText(value=appliance.duration_min, min=15, max=720, step=15, description="Runtime (min):"),
            "earliest_min": widgets.BoundedIntText(value=appliance.earliest_min, min=0, max=720, step=15, description="Earliest (min):"),
            "deadline_min": widgets.BoundedIntText(value=appliance.deadline_min, min=15, max=720, step=15, description="Finish by (min):")}
        for control in options.values():
            control.layout.width = "245px"
            if hasattr(control, "style"):
                control.style.description_width = "105px"
        grid = widgets.GridBox(list(options.values()), layout=widgets.Layout(
            grid_template_columns="repeat(3, minmax(245px, 1fr))", grid_gap="6px", width="100%", overflow="auto"))
        appliance_sections.append(grid)
        appliance_forms.append({"initial": initial, "controls": options})
    advanced = widgets.Accordion(children=appliance_sections)
    for i, appliance in enumerate(default_appliances()):
        advanced.set_title(i, appliance.name)
    appliance_help = widgets.HTML("<p>Expand an appliance to change its settings. Earliest start and finish deadline are minutes "
        "after the selected origin, in 15-minute steps. Defaults are scenario examples. Essential appliances remain protected.</p>")
    explain = widgets.Checkbox(value=True, description="Show SHAP and LSTM activations", indent=False)
    button = widgets.Button(description="Run optimizer", button_style="success", icon="play", layout=widgets.Layout(width="180px"))
    status, result_output = widgets.HTML(), widgets.Output()
    state = {"last_result": None, "last_path": None}

    def run(_=None):
        button.disabled = True
        status.value = "<p>Calculating forecast and schedule…</p>"
        try:
            appliance_values = [Appliance(**dict(form["initial"], **{key: control.value for key, control in form["controls"].items()}))
                                for form in appliance_forms]
            scenario = Scenario(occupancy=occupancy.value, temperature_c=temperature.value,
                peak_start_hour=peak_start.value, peak_end_hour=peak_end.value, peak_rate=peak_rate.value,
                offpeak_rate=offpeak_rate.value, horizon_hours=horizon.value, max_parallel_jobs=concurrency.value)
            forecast = bridge.predict(case.value)
            result = optimize_decisions(forecast, appliance_values, scenario, knowledge)
            explanation = bridge.explain(case.value) if explain.value else None
            folder = Path(output_dir) / datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%S_%fZ")
            folder.mkdir(parents=True, exist_ok=False)
            report = dict(result, task7_run_id=bridge.manifest.get("run_id"), hybrid_signature=bridge.signature)
            if explanation is not None:
                report["ml_shap"] = {"base_kw": explanation["base"], "raw_forecast_kw": explanation["raw_prediction"],
                                     "features": explanation["features"], "values_kw": explanation["values"].tolist()}
            (folder / "optimizer_result.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
            pd.DataFrame(result["recommendations"]).to_csv(folder / "recommendations.csv", index=False)
            pd.DataFrame(result["plan"]["schedule"]).to_csv(folder / "schedule.csv", index=False)
            with result_output:
                clear_output(wait=True)
                render_optimizer_result(result, bridge, explanation)
                if performance is not None:
                    display(HTML("<details><summary><b>Forecasting performance on the full saved test period</b></summary></details>"))
                    display(performance.round(4))
                print("Saved this scenario:", folder.resolve())
            state["last_result"], state["last_path"] = result, folder
            status.value = "<p style='color:#126b46'><b>Optimizer complete.</b> Change settings and run again to compare scenarios.</p>"
        except Exception as error:
            status.value = f"<p style='color:#b42318'><b>Inputs need attention:</b> {escape(str(error))}</p>"
            with result_output:
                clear_output(wait=True)
                print("No new result was produced. Review the message above.")
        finally:
            button.disabled = False

    button.on_click(run)
    header = widgets.HTML("<h2 style='color:#176b87'>Smart Electricity Consumption Optimizer</h2>"
        "<p><b>Task 9:</b> LSTM patterns → ML forecast → ontology/rules → A* schedule → explanation.</p>"
        "<p>Choose a recorded electricity case and enter a household scenario. Tariffs and appliance settings below are editable examples. "
        "The forecast follows the selected electricity history; household settings affect decisions and estimated costs. "
        "This dashboard provides recommendations; it does not switch hardware.</p>")
    controls = widgets.VBox([header, case,
        widgets.HBox([occupancy, temperature]), widgets.HBox([peak_rate, offpeak_rate]),
        widgets.HBox([peak_start, peak_end]), widgets.HBox([horizon, concurrency]), appliance_help, advanced, explain,
        widgets.HBox([button, status]), result_output], layout=widgets.Layout(width="100%"))
    return {"widget": controls, "run": run, "state": state,
            "controls": {"case": case, "occupancy": occupancy, "temperature": temperature,
                         "appliances": appliance_forms, "explain": explain, "peak_rate": peak_rate,
                         "offpeak_rate": offpeak_rate, "horizon": horizon, "concurrency": concurrency}}

In [ ]:
if SHOW_DASHBOARD:
    evaluation_dashboard = build_optimizer_dashboard(bridge, knowledge, OUTPUT_DIR_10 / "demonstration", metrics)
    display(evaluation_dashboard["widget"])
    evaluation_dashboard["run"]()
else:
    print("Dashboard display skipped; evaluation and reports remain available.")

## 10. Final audit, report and presentation material
Fingerprints must match before/after evaluation. The report preserves failed
checks and weak results rather than hiding them. It includes the architecture,
methodology, actual metrics, explanation scope, responsible-AI discussion and
limitations. Presentation notes organize these into eight talking points.

In [ ]:
frozen_after = freeze_fingerprint(bridge, knowledge)
if frozen_before != frozen_after:
    raise RuntimeError("A frozen model/ontology fingerprint changed during evaluation. Investigate before reporting results.")
report_text = write_evaluation_documents(OUTPUT_DIR_10, bridge, metrics, scenario_checks, fairness_checks, xai,
    freshness, intervals_note, setup_mode, frozen_before)
audit = {"evaluation_completed": True, "models_unchanged_during_evaluation": True, "setup_mode": setup_mode,
    "scenario_checks_passed": int(scenario_checks["Passed"].sum()), "scenario_checks_total": len(scenario_checks),
    "shap_checks_passed": int(xai["checks"]["Passed"].sum()), "shap_checks_total": len(xai["checks"]),
    "policy_invariance_checks_passed": int(fairness_checks["Passed"].sum()), "policy_invariance_checks_total": len(fairness_checks),
    "external_csv_supplied": bool(UNSEEN_MINUTE_CSV), "reused_holdout": True, "task7_run": bridge.manifest["run_id"],
    "versions": {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__}}
(OUTPUT_DIR_10 / "evaluation_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
print(report_text)
print("Model fingerprints unchanged:", frozen_before == frozen_after)
print("All Task 10 tables, charts, report, model card, deployment guide and presentation notes:", OUTPUT_DIR_10.resolve())
print("Save this executed notebook and inspect any failed scenario/explanation checks before submission.")